In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import json
import pickle
import datetime
from pytz import timezone

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [2]:
import traceback
from IPython.core.magic import register_line_magic
from IPython.display import display, Markdown
from IPython import get_ipython
import os
import pandas as pd
import numpy as np

# def custom_error_handler(shell, etype, evalue, tb, tb_offset=None):
#     error_message = ''.join(traceback.format_exception(etype, evalue, tb))

#     # Display the error message in the notebook
#     display(Markdown(f"**Error Occurred in Cell:**\n```\n{error_message}\n```"))

#     # Ensure file is created and writable
#     import requests
#     teleTOKEN = "6985416199:AAE0DFG5tsV9Dc5kBGE4fp-bze7Lc3NPPIY"
#     chat_id = "-4100368104"
#     message = "🚨🚨 Drive Connector NB Failed 🚨🚨"
#     url = f"https://api.telegram.org/bot{teleTOKEN}/sendMessage?chat_id={chat_id}&text={message}"
#     print(requests.get(url).json()) # this sends the message

# Register the custom handler
# ip = get_ipython()
# ip.set_custom_exc((Exception,), custom_error_handler)

import os
import time

os.environ["TZ"] = 'Asia/Kolkata'
time.tzset()

import sys
import requests

def download_file_from_google_drive(id, destination):
    URL = "https://docs.google.com/uc?export=download&confirm=1"

    session = requests.Session()

    response = session.get(URL, params={"id": id}, stream=True)
    token = get_confirm_token(response)

    if token:
        params = {"id": id, "confirm": token}
        response = session.get(URL, params=params, stream=True)

    save_response_content(response, destination)


def get_confirm_token(response):
    for key, value in response.cookies.items():
        if key.startswith("download_warning"):
            return value

    return None


def save_response_content(response, destination):
    CHUNK_SIZE = 32768

    with open(destination, "wb") as f:
        for chunk in response.iter_content(CHUNK_SIZE):
            if chunk:  # filter out keep-alive new chunks
                f.write(chunk)

download_file_from_google_drive("10QPeNalqUU39Kd9osjjKiT_jJK8RG3A7", 'PyDrive2-1.14.0-py3-none-any.whl')

!pip install 'PyDrive2-1.14.0-py3-none-any.whl'

import json

download_file_from_google_drive("1xW5PG8mHNKO0-KoicDqEQuEbrkoabKeJ", 'nifty_50_100_200_500.csv')
download_file_from_google_drive("116-hAUPbtBIyCyRAwgs3IpxCIhUABKso", 'client_secrets.json')
download_file_from_google_drive("1OZTyQLTnPA3B-PHAaD-InCDRGaCBDSk_", 'credentials.json')
download_file_from_google_drive("1OFIv9ciq4oD_YyzyOxiShAOk-ITGxyxI", 'zerodha_creds.json')
download_file_from_google_drive("10MTP0t80facteS5WDVspO5Ea5bA-x8ig", 'breakEvenList.json')

with open("breakEvenList.json", 'r') as f:
    breakEvenList = json.load(f)

from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

gauth = GoogleAuth()
credential_file = "credentials.json"

gauth.LoadCredentialsFile(credential_file)
gauth.LocalWebserverAuth() # client_secrets.json need to be in the same directory as the script
drive = GoogleDrive(gauth)

def downloadFromDrive(file_id):
    f = drive.CreateFile({'id': file_id})
    f.FetchMetadata()
    f.GetContentFile(f["title"])

print(os.getcwd())
print(os.listdir())

!pip install 'PyDrive2-1.14.0-py3-none-any.whl'

Processing ./PyDrive2-1.14.0-py3-none-any.whl
  Attempting uninstall: PyDrive2
    Found existing installation: PyDrive2 1.21.3
    Uninstalling PyDrive2-1.21.3:
      Successfully uninstalled PyDrive2-1.21.3
/kaggle/working
['nifty_50_100_200_500.csv', '.virtual_documents', 'credentials.json', 'PyDrive2-1.14.0-py3-none-any.whl', 'client_secrets.json', 'breakEvenList.json', 'zerodha_creds.json']
Processing ./PyDrive2-1.14.0-py3-none-any.whl
PyDrive2 is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.


In [3]:
# Sample
downloadFromDrive('1hY6E15NLoLqrzG0R76tax9kkw86j6Gcb')

In [4]:
df = pd.read_csv('/kaggle/working/daysPNL_dfUpdated_fixed.csv')
df.head()

,date,overallPNL,PNLonFundsChurned,totalAccountValue,totalIdleFund,percIdleFund,percPNL_onHoldingFund,niftyReturns,sensexReturns,niftyBankReturns,niftyMidCapReturns,niftySmallCapReturns
0,2024-09-04 13:51:19,0.050,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-09-05 15:30:06,0.004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-09-06 15:30:07,0.117,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-09-09 15:29:37,0.063,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-09-10 15:29:58,0.140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
df.shape

(544, 12)

In [6]:
if df.columns[0].isdigit() or df.columns[0] == "":
    df = df.iloc[:, 1:]

# df["date"] = pd.to_datetime(df["date"])
df["date"] = pd.to_datetime(df["date"], format="mixed")
df = df.sort_values("date").reset_index(drop=True)

In [7]:
df.tail(2)

,date,overallPNL,PNLonFundsChurned,totalAccountValue,totalIdleFund,percIdleFund,percPNL_onHoldingFund,niftyReturns,sensexReturns,niftyBankReturns,niftyMidCapReturns,niftySmallCapReturns
542,2026-08-06 15:35:21.193872,0.0,0.0,638763.189988,360994.2,56.515,0.0,0.046,0.476,0.561,-0.397,0.200
543,2026-08-07 15:35:23.286474,0.0,0.0,638763.189988,320770.3,50.217,0.0,-0.265,-0.577,-0.546,0.225,-0.011


In [8]:
!pip install kiteconnect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 657.4 kB/s eta 0:00:00 0:00:01
INFO: pip is looking at multiple versions of service-identity to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.5/771.5 kB 4.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 26.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.7/270.7 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 4.6 MB/s eta 0:00:00


In [9]:
downloadFromDrive('1gdtv9Eh7pKuE94FbgfurnVs3xZ3hCrTx') #Webdriver Manager Wheeler File
downloadFromDrive('12WNzoM-xf5q_RoWrxNbWi7EY9mh2t4zm') #Talib Wheeler File
downloadFromDrive('1-idn2TLkBUUamg0LL-Jn7ZB-JQVKkZ0F') #Selenium Wheeler File
downloadFromDrive('1L4Ofg62x4oBiJo3EzBFUXpNfWLy9AJo0') #Kiteconnect Wheeler File
downloadFromDrive('1ULiCBQwtAk9XetnmnwfnUt2MbUIGsWzP') #Chromedriver Wheeler File

In [10]:
downloadFromDrive('1P0OXwTFzAo5eERHeeVdPl5ujBvU-kYWL')

In [11]:
!pip install 'webdriver_manager-3.8.3-py2.py3-none-any.whl'

Processing ./webdriver_manager-3.8.3-py2.py3-none-any.whl


In [12]:
!pip install 'selenium-4.4.3-py3-none-any.whl'
!pip install 'pyotp-2.7.0-py3-none-any.whl'
!pip install 'chromedriver_py-94.0.4606.41-py3-none-any.whl'

Processing ./selenium-4.4.3-py3-none-any.whl
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 685.3 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 4.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 9.3 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
id 1.6.1 requires urllib3<3,>=2, but you have urllib3 1.26.20 which is incompatible.
blobfile 3.2.0 requires urllib3>=2, but you have urllib3 1.26.20 which is incompatible.
Processing ./pyotp-2.7.0-py3-none-any.whl
Processing ./chromedriver_py-94.0.4606.41-py3-none-any.whl


In [13]:
!sudo apt-key adv --keyserver keyserver.ubuntu.com --recv-keys 4EB27DB2A3B88B8B
!sudo apt-key adv --keyserver keyserver.ubuntu.com --recv-keys B53DC80D13EDEF05

!sudo apt-get update --allow-releaseinfo-change

Executing: /tmp/apt-key-gpghome.sUjQi8JCEl/gpg.1.sh --keyserver keyserver.ubuntu.com --recv-keys 4EB27DB2A3B88B8B
gpg: key 7721F63BD38B4796: 2 duplicate signatures removed
gpg: key 7721F63BD38B4796: public key "Google Inc. (Linux Packages Signing Authority) <linux-packages-keymaster@google.com>" imported
gpg: Total number processed: 1
gpg:               imported: 1
Executing: /tmp/apt-key-gpghome.rSdiSPm67e/gpg.1.sh --keyserver keyserver.ubuntu.com --recv-keys B53DC80D13EDEF05
gpg: key B53DC80D13EDEF05: 1 duplicate signature removed
gpg: key B53DC80D13EDEF05: public key "Rapture Automatic Signing Key (cloud-rapture-signing-key-2022-03-07-08_01_01.pub)" imported
gpg: Total number processed: 1
gpg:               imported: 1
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [105 kB]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]               
Get:4 https://cli.gi

In [14]:
!DEBIAN_FRONTEND=noninteractive apt update
!apt-get --yes -o Dpkg::Options::="--force-confdef" -o Dpkg::Options::="--force-confold" upgrade

!apt-get update

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease                         
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease             
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease    m
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease          
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
198 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to pr

In [15]:
!apt-get install -y gcc python3-dev wget gconf-service libasound2 libatk1.0-0 libcairo2 libcups2 libfontconfig1 libgdk-pixbuf2.0-0 libgtk-3-0 libnspr4 libpango-1.0-0 libxss1 fonts-liberation libappindicator1 libnss3 xdg-utils libgbm1

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-liberation is already the newest version (1:1.07.4-11).
gcc is already the newest version (4:11.2.0-1ubuntu1).
gcc set to manually installed.
libfontconfig1 is already the newest version (2.13.1-4.2ubuntu5).
libxss1 is already the newest version (1:1.2.3-1build2).
libxss1 set to manually installed.
libasound2 is already the newest version (1.2.6.1-1ubuntu1.2).
libasound2 set to manually installed.
libcairo2 is already the newest version (1.16.0-5ubuntu2.1).
libcairo2 set to manually installed.
libcups2 is already the newest version (2.4.1op1-1ubuntu4.21).
libcups2 set to manually installed.
libgbm1 is already the newest version (23.2.1-1ubuntu3.1~22.04.4).
libgbm1 set to manually installed.
libnspr4 is already the newest version (2:4.35-0ubuntu0.22.04.1).
libnspr4 set to manually installed.
libnss3 is already the newest version (2:3.98-0ubuntu0.22.04.4).
libnss3 set to manually instal

In [16]:
downloadFromDrive('12QTXe4AAKJ2dpkguvtBpSj7J-rxZgTI7')
!dpkg -i google-chrome-stable_94.0.4606.71-1_amd64.deb; apt-get -fy install

Selecting previously unselected package google-chrome-stable.
(Reading database ... 121607 files and directories currently installed.)
Preparing to unpack google-chrome-stable_94.0.4606.71-1_amd64.deb ...
Unpacking google-chrome-stable (94.0.4606.71-1) ...
Setting up google-chrome-stable (94.0.4606.71-1) ...
update-alternatives: using /usr/bin/google-chrome-stable to provide /usr/bin/x-www-browser (x-www-browser) in auto mode
update-alternatives: using /usr/bin/google-chrome-stable to provide /usr/bin/gnome-www-browser (gnome-www-browser) in auto mode
update-alternatives: using /usr/bin/google-chrome-stable to provide /usr/bin/google-chrome (google-chrome) in auto mode
Processing triggers for mailcap (3.70+nmu1ubuntu1.22.04.1) ...
Processing triggers for man-db (2.10.2-1) ...
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [17]:
# !pip uninstall attrs --y
# !pip install attrs==21.4.0

In [18]:
# !pip install --force-reinstall --ignore-installed attrs==25.3.0

In [19]:
import pyotp
from kiteconnect import KiteTicker
from kiteconnect import KiteConnect

In [20]:
def add_driver_options(options):
    """
    Add configurable options
    """
    chrome_options = Options()
    for opt in options:
        chrome_options.add_argument(opt)
    return chrome_options

def initialize_driver():
    """
    Initialize the web driver
    """
    driver_config = {
        "executable_path": binary_path,
        "options": [
            "--headless",
            "--no-sandbox",
            "--start-fullscreen",
            "--allow-insecure-localhost",
            "--disable-dev-shm-usage",
        ],
    }
    options = add_driver_options(driver_config["options"])    
    driver = webdriver.Chrome(
        executable_path=driver_config["executable_path"], options=options
    )
    return driver

In [21]:
import time
import os
import datetime
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from chromedriver_py import binary_path
import urllib.parse as urlparse
import json
import pyotp
import random
import math

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

from kiteconnect import KiteTicker
from kiteconnect import KiteConnect

credsFile = 'zerodha_creds.json'
with open(credsFile,'r') as credsFile:
    creds = json.load(credsFile)
    
ZerodhaLoginName = creds['ZerodhaLoginName']
ZerodhaPass = creds['ZerodhaPass']
apiKey1 = creds['apiKey1']
apisec1 = creds['apisec1']
TOTP_key = creds['TOTP_key']

api_key = apiKey1
api_secret = apisec1

kite = KiteConnect(api_key = api_key)
url = (kite.login_url())

loginName = ZerodhaLoginName
password = ZerodhaPass

driver = initialize_driver()
driver.get(url)

time.sleep(5)

##################################### Login Page #########################################
form = WebDriverWait(driver,10).until(EC.visibility_of_element_located((By.XPATH,"//div[@class='login-form']")))
driver.find_element(By.XPATH ,"//input[@type='text']").send_keys(loginName)
driver.find_element(By.XPATH,"//input[@type='password']").send_keys(password)
time.sleep(5)
driver.find_element(By.XPATH,"//button[@type='submit']").click()
time.sleep(5)
print("passed")
##################################### PIN Page #############################################
form = WebDriverWait(driver,10).until(EC.visibility_of_element_located((By.XPATH,"//div[@class='login-form']")))
time.sleep(2)
TOTP_pin = pyotp.TOTP(TOTP_key) 
#driver.find_element(By.XPATH,"//input[@type='text']").send_keys(TOTP_pin.now())
driver.find_element(By.XPATH,"//input[@type='number']").send_keys(TOTP_pin.now())
time.sleep(2)
try:
    driver.find_element(By.XPATH,"//button[@type='submit']").click()
except:
    pass
time.sleep(10)
print('Request token url generated')
###################################### Session start and request token fetch ###########################################
session_url = driver.current_url
parsed_url = urlparse.urlparse(session_url)
# print(parsed_url)
request_token = urlparse.parse_qs(parsed_url.query)['request_token'][0]

/tmp/ipykernel_58/1669025317.py:25: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  driver = webdriver.Chrome(


passed
Request token url generated


In [22]:
kite = KiteConnect(api_key=api_key)

data = kite.generate_session(request_token, api_secret=api_secret)
access_token = data["access_token"]

kite.set_access_token(access_token)
kws = KiteTicker(api_key, access_token)

print(kite.ltp('NSE:RELIANCE'))

{'NSE:RELIANCE': {'instrument_token': 738561, 'last_price': 1334.8}}


In [23]:
import json

In [24]:


# from kiteconnect import KiteConnect
# import json

# with open("zerodha_creds.json") as f:
#     creds = json.load(f)

# kite = KiteConnect(api_key=creds["apiKey1"])
# print(kite.login_url())


In [25]:
# from kiteconnect import KiteConnect
# import json

# with open("zerodha_creds.json") as f:
#     creds = json.load(f)

# kite = KiteConnect(api_key=creds["apiKey1"])

# data = kite.generate_session(
#     request_token="3x3n2ACbhE628BpAnzTcLLHWCHoNVtsN",
#     api_secret=creds["apisec1"]
# )

# access_token = data["access_token"]

# kite.set_access_token(access_token)

# print(kite.profile())

In [26]:
symbols = [
    "NSE:NIFTY 50",
    "BSE:SENSEX",
    "NSE:NIFTY BANK",
    "NSE:NIFTY MIDCAP 150",
    "NSE:NIFTY SMLCAP 250"
]

ltp_data = kite.ltp(symbols)

for symbol, data in ltp_data.items():
    print(symbol, data["instrument_token"])

BSE:SENSEX 265
NSE:NIFTY 50 256265
NSE:NIFTY BANK 260105
NSE:NIFTY MIDCAP 150 266249
NSE:NIFTY SMLCAP 250 267273


In [27]:
# Fetch all mutual fund instruments
mf_instruments = kite.mf_instruments()

# Convert the list of dicts into a structured DataFrame
mf_df = pd.DataFrame(mf_instruments)

In [28]:
mf_df.head()

,tradingsymbol,amc,name,purchase_allowed,redemption_allowed,minimum_purchase_amount,purchase_amount_multiplier,minimum_additional_purchase_amount,minimum_redemption_quantity,redemption_quantity_multiplier,dividend_type,scheme_type,plan,settlement_type,last_price,last_price_date
0,INF00XX01135,ITI MUTUAL FUND_MF,ITI Multi Cap Fund,False,True,1000.0,1.0,1000.0,0.001,0.001,growth,Equity,regular,T2,26.0946,2026-08-07
1,INF00XX01143,ITI MUTUAL FUND_MF,ITI Multi Cap Fund,False,True,1000.0,1.0,1000.0,0.001,0.001,idcw-payout,Equity,regular,T2,22.8256,2026-08-07
2,INF00XX01168,ITI MUTUAL FUND_MF,ITI Multi Cap Fund - Direct Plan,True,True,1000.0,1.0,1000.0,0.001,0.001,growth,Equity,direct,T2,29.9538,2026-08-07
3,INF00XX01176,ITI MUTUAL FUND_MF,ITI Multi Cap Fund - Direct Plan,True,True,1000.0,1.0,1000.0,0.001,0.001,idcw-payout,Equity,direct,T2,26.4996,2026-08-07
4,INF00XX01192,ITI MUTUAL FUND_MF,ITI Liquid Fund,False,True,5000.0,1.0,1000.0,0.001,0.001,growth,Debt,regular,T1,1451.1114,2026-08-07


In [29]:
mf_df.loc[mf_df['name'].str.contains('Quant Small Cap Fund', case=False, na=False)]

,tradingsymbol,amc,name,purchase_allowed,redemption_allowed,minimum_purchase_amount,purchase_amount_multiplier,minimum_additional_purchase_amount,minimum_redemption_quantity,redemption_quantity_multiplier,dividend_type,scheme_type,plan,settlement_type,last_price,last_price_date
7482,INF966L01663,QUANTMUTUALFUND_MF,Quant Small Cap Fund - Direct Plan,True,True,5000.0,1.0,1000.0,0.001,0.001,idcw-payout,Equity,direct,T2,246.6891,2026-08-07
7483,INF966L01689,QUANTMUTUALFUND_MF,Quant Small Cap Fund - Direct Plan,True,True,5000.0,1.0,1000.0,0.001,0.001,growth,Equity,direct,T2,315.5842,2026-08-07
7496,INF966L01AA0,QUANTMUTUALFUND_MF,Quant Small Cap Fund,False,True,5000.0,1.0,1000.0,0.001,0.001,growth,Equity,regular,T2,287.8547,2026-08-07


In [30]:
mf_df.loc[mf_df['name'].str.contains('Motilal Oswal Midcap Fund', case=False, na=False)]

,tradingsymbol,amc,name,purchase_allowed,redemption_allowed,minimum_purchase_amount,purchase_amount_multiplier,minimum_additional_purchase_amount,minimum_redemption_quantity,redemption_quantity_multiplier,dividend_type,scheme_type,plan,settlement_type,last_price,last_price_date
3987,INF247L01411,MOTILALOSWAL_MF,Motilal Oswal Midcap Fund,False,True,500.0,1.0,500.0,0.001,0.001,growth,Equity,regular,T2,102.3896,2026-08-07
3988,INF247L01429,MOTILALOSWAL_MF,Motilal Oswal Midcap Fund,False,True,500.0,1.0,500.0,0.001,0.001,idcw-reinvest,Equity,regular,T2,46.2697,2026-08-07
3989,INF247L01437,MOTILALOSWAL_MF,Motilal Oswal Midcap Fund,False,True,500.0,1.0,500.0,0.001,0.001,idcw-payout,Equity,regular,T2,46.2697,2026-08-07
3990,INF247L01445,MOTILALOSWAL_MF,Motilal Oswal Midcap Fund - Direct Plan,True,True,500.0,1.0,500.0,0.001,0.001,growth,Equity,direct,T2,118.2721,2026-08-07
3991,INF247L01460,MOTILALOSWAL_MF,Motilal Oswal Midcap Fund - Direct Plan,True,True,500.0,1.0,500.0,0.001,0.001,idcw-payout,Equity,direct,T2,48.2405,2026-08-07


In [31]:
find_mf = mf_df.loc[(mf_df['dividend_type'] == 'growth') & (mf_df['scheme_type'] == 'Equity') & (mf_df['plan'] == 'direct')]


In [32]:
find_mf.shape

(590, 16)

In [33]:
find_mf.loc[find_mf['name'].str.contains('invesco india mid', case=False, na=False)]

,tradingsymbol,amc,name,purchase_allowed,redemption_allowed,minimum_purchase_amount,purchase_amount_multiplier,minimum_additional_purchase_amount,minimum_redemption_quantity,redemption_quantity_multiplier,dividend_type,scheme_type,plan,settlement_type,last_price,last_price_date
3426,INF205K01MV6,INVESCOMUTUALFUND_MF,Invesco India Midcap Fund - Direct Plan,True,True,100.0,1.0,100.0,0.001,0.001,growth,Equity,direct,T2,243.44,2026-08-07


In [34]:


MF_TOKENS = {
    "Parag_Parikh_Flexi_Cap_Fund": "INF879O01027",
    "QUANTMUTUALFUND_MF": "INF966L01721",
    "SBIMutualFund_MF" : "INF200K01UY4",
    "ICICIPrudentialMutualFund_MF" : "INF109K018M4",
    "BirlaSunLifeMutualFund_MF" : "INF209KB1O82",
    "INVESCOMUTUALFUND_MF" : "INF205K01NG5",
    "MOTILALOSWAL_MF" : "INF247L01445",

    "BANDHANMUTUALFUND_MF" : "INF194KB1AL4",
    "EDELWEISSMUTUALFUND_MF" : "INF843K01AO4",
    "HDFCMutualFund_MF" : "INF179K01XQ0",
    "INVESCOMUTUALFUND_MidCap_MF" : "INF205K01MV6",

}


In [35]:
MF_TOKENS

{'Parag_Parikh_Flexi_Cap_Fund': 'INF879O01027',
 'QUANTMUTUALFUND_MF': 'INF966L01721',
 'SBIMutualFund_MF': 'INF200K01UY4',
 'ICICIPrudentialMutualFund_MF': 'INF109K018M4',
 'BirlaSunLifeMutualFund_MF': 'INF209KB1O82',
 'INVESCOMUTUALFUND_MF': 'INF205K01NG5',
 'MOTILALOSWAL_MF': 'INF247L01445',
 'BANDHANMUTUALFUND_MF': 'INF194KB1AL4',
 'EDELWEISSMUTUALFUND_MF': 'INF843K01AO4',
 'HDFCMutualFund_MF': 'INF179K01XQ0',
 'INVESCOMUTUALFUND_MidCap_MF': 'INF205K01MV6'}

In [36]:
import requests
import pandas as pd

# Define a standard browser identity to bypass firewall blocking
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

def get_mf_history(isin_code):
    # Step 1: Download the complete scheme index map (MFapi doesn't search ISINs via query)
    print("Fetching master index...")
    index_url = "https://api.mfapi.in/mf"
    index_response = requests.get(index_url, headers=HEADERS).json()
    
    # Step 2: Extract the numeric schemeCode by matching your Zerodha ISIN
    # MFapi returns items as: {"schemeCode": 122639, "schemeName": "...", "isinGrowth": "INF879O01027", ...}
    scheme_code = None
    for scheme in index_response:
        if scheme.get('isinGrowth') == isin_code or scheme.get('isinDivReinvestment') == isin_code:
            scheme_code = scheme['schemeCode']
            break
            
    if not scheme_code:
        raise ValueError(f"ISIN {isin_code} could not be resolved on MFapi index.")
        
    print(f"Found Scheme Code: {scheme_code}. Fetching historical data...")
    
    # Step 3: Extract the time-series NAV matrix
    data_url = f"https://api.mfapi.in/mf/{scheme_code}"
    data_response = requests.get(data_url, headers=HEADERS).json()
    
    # Step 4: Parse cleanly into a Pandas dataset
    df = pd.DataFrame(data_response['data'])
    df.columns = ['Date', 'NAV']
    df['NAV'] = pd.to_numeric(df['NAV'])
    df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
    
    return df.sort_values(by='Date').reset_index(drop=True)

# Execute for Parag Parikh Flexi Cap Fund
df_ppfas = get_mf_history("INF879O01027")
print(df_ppfas.tail(5))


Fetching master index...
Found Scheme Code: 122639. Fetching historical data...
           Date      NAV
3241 2026-08-03  93.6064
3242 2026-08-04  93.3575
3243 2026-08-05  92.8329
3244 2026-08-06  92.3083
3245 2026-08-07  92.3269


In [37]:
import pandas as pd
mf_returns_df = None

for col, token in MF_TOKENS.items():

    mf_hist = get_mf_history(token)

    mf_hist["trade_date"] = (
        pd.to_datetime(mf_hist["Date"])
          .dt.tz_localize(None)
          .dt.normalize()
    )

    mf_hist[col] = mf_hist["NAV"].pct_change() * 100

    mf_hist = mf_hist[["trade_date", col]]

    if mf_returns_df is None:
        mf_returns_df = mf_hist
    else:
        mf_returns_df = mf_returns_df.merge(mf_hist, on="trade_date", how="outer")

Fetching master index...
Found Scheme Code: 122639. Fetching historical data...
Fetching master index...
Found Scheme Code: 120833. Fetching historical data...
Fetching master index...
Found Scheme Code: 119732. Fetching historical data...
Fetching master index...
Found Scheme Code: 120621. Fetching historical data...
Fetching master index...
Found Scheme Code: 147844. Fetching historical data...
Fetching master index...
Found Scheme Code: 120395. Fetching historical data...
Fetching master index...
Found Scheme Code: 127042. Fetching historical data...
Fetching master index...
Found Scheme Code: 147946. Fetching historical data...
Fetching master index...
Found Scheme Code: 140228. Fetching historical data...
Fetching master index...
Found Scheme Code: 118989. Fetching historical data...
Fetching master index...
Found Scheme Code: 120403. Fetching historical data...


In [38]:
mf_returns_df.shape

(3366, 12)

In [39]:
mf_returns_df = mf_returns_df.loc[mf_returns_df['trade_date'] > '2024-01-01']

In [40]:
mf_returns_df.shape

(649, 12)

In [41]:
INDEX_TOKENS = {
    "niftyReturns": 256265,
    "sensexReturns": 265,
    "niftyBankReturns": 260105,
    "niftyMidCapReturns": 266249,
    "niftySmallCapReturns": 267273

}

In [42]:
import pandas as pd

start_date = "2024-01-01"
end_date = "2026-12-31"

returns_df = None

for col, token in INDEX_TOKENS.items():

    hist = pd.DataFrame(
        kite.historical_data(
            instrument_token=token,
            from_date=start_date,
            to_date=end_date,
            interval="day"
        )
    )

    hist["trade_date"] = (
        pd.to_datetime(hist["date"])
          .dt.tz_localize(None)
          .dt.normalize()
    )
    hist[col] = hist["close"].pct_change()

    hist = hist[["trade_date", col]]

    if returns_df is None:
        returns_df = hist
    else:
        returns_df = returns_df.merge(hist, on="trade_date", how="outer")

In [43]:
returns_df.head()

,trade_date,niftyReturns,sensexReturns,niftyBankReturns,niftyMidCapReturns,niftySmallCapReturns
0,2024-01-01,NaN,NaN,NaN,NaN,NaN
1,2024-01-02,-0.003500,-0.005250,-0.009799,-0.001508,-0.000386
2,2024-01-03,-0.006852,-0.007454,-0.001187,0.002441,0.001186
3,2024-01-04,0.006564,0.006881,0.010290,0.014764,0.010275
4,2024-01-05,0.002410,0.002486,-0.000765,0.003205,0.006727


In [44]:
# df["trade_date"] = pd.to_datetime(df["date"]).dt.normalize()
df["date"] = pd.to_datetime(df["date"], format="mixed")
df["trade_date"] = pd.to_datetime(df["date"]).dt.normalize()

## merge with nifty
df = df.merge(
    returns_df,
    on="trade_date",
    how="left"
)

## merge with MF

df = df.merge(
    mf_returns_df,
    on="trade_date",
    how="left"
)



In [45]:
df.head()

,date,overallPNL,PNLonFundsChurned,totalAccountValue,totalIdleFund,percIdleFund,percPNL_onHoldingFund,niftyReturns_x,sensexReturns_x,niftyBankReturns_x,...,QUANTMUTUALFUND_MF,SBIMutualFund_MF,ICICIPrudentialMutualFund_MF,BirlaSunLifeMutualFund_MF,INVESCOMUTUALFUND_MF,MOTILALOSWAL_MF,BANDHANMUTUALFUND_MF,EDELWEISSMUTUALFUND_MF,HDFCMutualFund_MF,INVESCOMUTUALFUND_MidCap_MF
0,2024-09-04 13:51:19,0.050,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.456558,-1.106998,-0.093471,-0.633553,-0.112416,-0.254689,0.223075,-0.008694,-0.121108,0.147726
1,2024-09-05 15:30:06,0.004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.142595,-0.187424,0.154372,-0.255037,-0.350131,0.194552,0.667734,0.174761,0.766348,0.417091
2,2024-09-06 15:30:07,0.117,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.394029,-2.310376,-0.976179,-2.147788,-2.158364,-0.514196,-1.024050,-1.294970,-0.870864,-0.795259
3,2024-09-09 15:29:37,0.063,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.148791,-0.808089,-0.476393,-0.705513,-1.154290,0.039515,-0.591786,-0.362284,-0.478841,-0.224662
4,2024-09-10 15:29:58,0.140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.910216,0.200513,0.672986,0.236842,0.077851,1.248150,1.084171,0.720142,1.183577,1.115603


In [46]:
# df['PNLonFundsChurned'].sum()
# df['totalAccountValue'].sum()
# df['totalIdleFund'].sum()
# df['percIdleFund'].sum()
df['percPNL_onHoldingFund'].sum()


np.float64(4.822)

In [47]:
cols = [
    "niftyReturns",
    "sensexReturns",
    "niftyBankReturns",
    "niftyMidCapReturns",
    "niftySmallCapReturns"
]

for col in cols:
    df[col] = df[f"{col}_y"] * 100

df.drop(
    columns=[
        "trade_date",
        *[f"{c}_x" for c in cols],
        *[f"{c}_y" for c in cols]
    ],
    inplace=True
)


In [48]:
df.head()

,date,overallPNL,PNLonFundsChurned,totalAccountValue,totalIdleFund,percIdleFund,percPNL_onHoldingFund,Parag_Parikh_Flexi_Cap_Fund,QUANTMUTUALFUND_MF,SBIMutualFund_MF,...,MOTILALOSWAL_MF,BANDHANMUTUALFUND_MF,EDELWEISSMUTUALFUND_MF,HDFCMutualFund_MF,INVESCOMUTUALFUND_MidCap_MF,niftyReturns,sensexReturns,niftyBankReturns,niftyMidCapReturns,niftySmallCapReturns
0,2024-09-04 13:51:19,0.050,NaN,NaN,NaN,NaN,NaN,-0.237925,-0.456558,-1.106998,...,-0.254689,0.223075,-0.008694,-0.121108,0.147726,-0.321007,-0.245653,-0.558822,-0.084194,0.308252
1,2024-09-05 15:30:06,0.004,NaN,NaN,NaN,NaN,NaN,0.027840,-0.142595,-0.187424,...,0.194552,0.667734,0.174761,0.766348,0.417091,-0.212709,-0.183941,0.141634,0.372454,0.953844
2,2024-09-06 15:30:07,0.117,NaN,NaN,NaN,NaN,NaN,-1.267506,-1.394029,-2.310376,...,-0.514196,-1.024050,-1.294970,-0.870864,-0.795259,-1.165038,-1.237489,-1.741105,-1.370075,-1.013768
3,2024-09-09 15:29:37,0.063,NaN,NaN,NaN,NaN,NaN,-0.100776,-0.148791,-0.808089,...,0.039515,-0.591786,-0.362284,-0.478841,-0.224662,0.339005,0.462665,1.069560,-0.250971,-0.829972
4,2024-09-10 15:29:58,0.140,NaN,NaN,NaN,NaN,NaN,0.701676,0.910216,0.200513,...,1.248150,1.084171,0.720142,1.183577,1.115603,0.419868,0.443541,0.302243,1.073705,1.250833


In [49]:
df.to_csv('/kaggle/working/daysPNL_dfUpdated_fixed.csv', index=False)

In [50]:
# filepath = '/kaggle/working/daysPNL_dfUpdated_fixed.csv'
# output_path = "investment_dashboard.html"

# print(f"Loading: {filepath}")
# df = load_data(filepath)

In [51]:
print("hi")

hi


In [52]:
"""
investment_metrics.py
---------------------
Calculate portfolio evaluation metrics and generate an HTML dashboard.

Usage:
    python investment_metrics.py portfolio_data.csv
    python investment_metrics.py portfolio_data.csv my_dashboard.html

Expected columns (tab- or comma-separated):
    date, overallPNL, PNLonFundsChurned, totalAccountValue,
    totalIdleFund, percIdleFund, percPNL_onHoldingFund,
    niftyReturns, sensexReturns, niftyBankReturns,
    niftyMidCapReturns, niftySmallCapReturns

Requires: pandas, numpy  (pip install pandas numpy)
"""

import sys
import json
import numpy as np
import pandas as pd
from datetime import datetime


# ---------------------------------------------------------------------------
# 1. Data loading
# ---------------------------------------------------------------------------

def load_data(filepath: str) -> pd.DataFrame:
    sep = "\t" if filepath.endswith(".tsv") else ","
    df = pd.read_csv(filepath, sep=sep)
    df = _drop_index_column(df)
    df = _parse_date_column(df)
    df = df.sort_values("date").reset_index(drop=True)
    return df


def _drop_index_column(df: pd.DataFrame) -> pd.DataFrame:
    """Remove a leading integer-index column that pandas or Excel sometimes writes."""
    first = str(df.columns[0]).strip()
    # Covers: "", "Unnamed: 0", "0", pure-digit names
    if first in ("", "Unnamed: 0") or first.isdigit():
        df = df.iloc[:, 1:]
    # Also drop if first col has no name and its values are sequential integers
    elif pd.api.types.is_integer_dtype(df.iloc[:, 0]):
        expected = pd.Series(range(len(df)))
        if (df.iloc[:, 0].reset_index(drop=True) == expected).all():
            df = df.iloc[:, 1:]
    return df.reset_index(drop=True)


# def _parse_date_column(df: pd.DataFrame) -> pd.DataFrame:
#     """Parse the date column robustly; try common formats before giving up."""
#     col = df["date"]
#     # Already datetime — nothing to do
#     if pd.api.types.is_datetime64_any_dtype(col):
#         return df
#     # Try pandas auto-inference first (handles ISO formats well)
#     parsed = pd.to_datetime(col, errors="coerce", infer_datetime_format=True)
#     if parsed.isna().all():
#         # Fallback: try explicit Indian/common formats
#         for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d", "%d-%m-%Y", "%d/%m/%Y"):
#             parsed = pd.to_datetime(col, format=fmt, errors="coerce")
#             if not parsed.isna().all():
#                 break
#     n_bad = parsed.isna().sum()
#     if n_bad:
#         print(f"  Warning: {n_bad} date value(s) could not be parsed and will be dropped.")
#         df = df[parsed.notna()].copy()
#         parsed = parsed[parsed.notna()]
#     df["date"] = parsed.values
#     return df

def _parse_date_column(df):

    df["date"] = pd.to_datetime(
        df["date"],
        format="mixed",
        errors="coerce"
    )

    n_bad = df["date"].isna().sum()

    if n_bad:
        print(
            f"Warning: {n_bad} date value(s) could not be parsed."
        )

    df = df.dropna(subset=["date"])

    return df

# ---------------------------------------------------------------------------
# 2. Metric calculations
# ---------------------------------------------------------------------------

def _safe_mean(series: pd.Series):
    s = series.dropna()
    return float(s.mean()) if len(s) else None


def _safe_last(series: pd.Series):
    s = series.dropna()
    return float(s.iloc[-1]) if len(s) else None


def calculate_metrics(df: pd.DataFrame) -> dict:
    m = {}
    pnl = df["overallPNL"].dropna()

    # --- Absolute return --------------------------------------------------
    # m["total_pnl"]         = float(pnl.sum())
    m["total_pnl"] = (
        (1 + pnl / 100).prod() - 1
    ) * 100
    m["avg_daily_pnl"]     = float(pnl.mean()) if len(pnl) else None
    m["latest_pnl"]        = float(pnl.iloc[-1]) if len(pnl) else None

    hold = df.get("percPNL_onHoldingFund", pd.Series(dtype=float))
    m["avg_return_on_holding"]    = _safe_mean(hold)
    m["latest_return_on_holding"] = _safe_last(hold)

    churned = df.get("PNLonFundsChurned", pd.Series(dtype=float))
    mask = churned.notna() & pnl.notna() & (pnl != 0)
    if mask.sum():
        m["avg_churn_efficiency"] = float((churned[mask] / pnl[mask]).mean())
    else:
        m["avg_churn_efficiency"] = None

    # --- Alpha vs benchmarks ----------------------------------------------
    
    benchmarks = {
        "nifty": "niftyReturns",
        "sensex": "sensexReturns",
        "nifty_bank": "niftyBankReturns",
        "nifty_midcap": "niftyMidCapReturns",
        "nifty_smallcap": "niftySmallCapReturns",
    }
    
    alpha = {}
    
    # Strategy cumulative return
    strategy_return = (
        (1 + pnl / 100).prod() - 1
    ) * 100
    
    for label, col in benchmarks.items():
    
        if col not in df.columns:
            continue
    
        benchmark_series = df[col].dropna()
    
        if len(benchmark_series) == 0:
            continue
    
        benchmark_return = (
            (1 + benchmark_series / 100).prod() - 1
        ) * 100
    
        alpha[label] = round(
            strategy_return - benchmark_return,
            2
        )
    
    m["strategy_return"] = round(strategy_return, 2)
    m["alpha"] = alpha

    # --- Capital efficiency -----------------------------------------------
    idle_pct = df.get("percIdleFund", pd.Series(dtype=float))
    m["avg_idle_pct"] = _safe_mean(idle_pct)
    m["latest_idle_pct"] = _safe_last(idle_pct)
    
    m["avg_utilization"] = (
        100 - _safe_mean(idle_pct)
    ) if _safe_mean(idle_pct) is not None else None

    idle_fund = df.get("totalIdleFund", pd.Series(dtype=float))
    m["latest_idle_fund"] = _safe_last(idle_fund)

    if {"totalAccountValue", "totalIdleFund", "overallPNL"}.issubset(df.columns):
        vm = df["totalAccountValue"].notna() & df["totalIdleFund"].notna() & pnl.notna()
        if vm.sum():
            deployed = (df.loc[vm, "totalAccountValue"] - df.loc[vm, "totalIdleFund"]).replace(0, np.nan)
            m["return_per_deployed"] = float((pnl[vm] / deployed).mean())
    else:
        m["return_per_deployed"] = None

    # --- Risk metrics -----------------------------------------------------
    if len(pnl) > 1:
        diffs = pnl.diff().dropna()
        # m["daily_pnl_volatility"] = float(diffs.std())
        
        m["daily_pnl_volatility"] = float(
            pnl.std()
        )

        # cumulative = pnl.cumsum()
        # rolling_max = cumulative.cummax()
        # drawdown = cumulative - rolling_max
        
        equity = (
            100 * (1 + pnl / 100).cumprod()
        )
        
        rolling_max = equity.cummax()
        
        drawdown = (
            equity / rolling_max - 1
        ) * 100
        
        m["max_drawdown"] = drawdown.min()
        
        m["max_drawdown"] = float(drawdown.min())

        m["win_rate"] = float((pnl > 0).mean())

        if m["daily_pnl_volatility"] and m["daily_pnl_volatility"] != 0:
            m["sharpe_like"] = float(m["avg_daily_pnl"] / m["daily_pnl_volatility"])
        else:
            m["sharpe_like"] = None
    else:
        m["daily_pnl_volatility"] = m["max_drawdown"] = m["win_rate"] = m["sharpe_like"] = None

    # --- Trend ------------------------------------------------------------
    if len(pnl) >= 7:
        m["rolling_7d_pnl"] = float(pnl.rolling(7).mean().iloc[-1])
    else:
        m["rolling_7d_pnl"] = None

    df2 = df.copy()
    df2["month"] = df2["date"].dt.to_period("M")
    # monthly = df2.groupby("month")["overallPNL"].sum()
    monthly = (
        df2.groupby("month")["overallPNL"]
           .apply(lambda x: ((1 + x/100).prod() - 1) * 100)
    )
    
    m["monthly_pnl"] = {str(k): round(v, 6) for k, v in monthly.items()}
    if len(monthly) >= 2:
        m["mom_change"] = float(monthly.iloc[-1] - monthly.iloc[-2])
    else:
        m["mom_change"] = None

    return m


# ---------------------------------------------------------------------------
# 3. Terminal summary
# ---------------------------------------------------------------------------

def print_summary(m: dict):
    def row(label, val, pct=False, already_pct=False, higher_good=True):
        if val is None:
            print(f"  {label:<40}  {'N/A':>12}")
            return
        if already_pct:
            fval = f"{val:.2f}%"
        elif pct:
            fval = f"{val*100:.2f}%"
        else:
            fval = f"{val:+.4f}"
            
        # fval = f"{val*100:.2f}%" if pct else f"{val:+.4f}"
        ok = (val > 0) == higher_good
        mark = "✓" if ok else "✗"
        print(f"  {label:<40}  {fval:>12}  {mark}")

    print("\n" + "=" * 60)
    print("  PORTFOLIO METRICS SUMMARY")
    print("=" * 60)

    print("\n  RETURN PERFORMANCE")
    row("Compounded Total Return (%)",              m.get("total_pnl"))
    row("Avg daily P&L",               m.get("avg_daily_pnl"))
    row("Return on holding fund (avg)", m.get("avg_return_on_holding"))
    row("Churn efficiency ratio",      m.get("avg_churn_efficiency"))

    print("\n  BENCHMARK ALPHA")
    for k, v in (m.get("alpha") or {}).items():
        row(f"Alpha vs {k.replace('_', ' ').title()}", v)

    print("\n  CAPITAL EFFICIENCY")
    # row("Avg capital utilization",  m.get("avg_utilization"), pct=True)
    # row("Avg idle fund %",          m.get("avg_idle_pct"),    pct=True, higher_good=False)
    

    row("Avg capital utilization",
        m.get("avg_utilization"),
        already_pct=True)
    
    row("Avg idle fund %",
        m.get("avg_idle_pct"),
        already_pct=True,
        higher_good=False)

    row("Return per ₹ deployed",    m.get("return_per_deployed"))

    print("\n  RISK METRICS")
    row("Win rate",           m.get("win_rate"),              pct=True)
    row("Max drawdown",       m.get("max_drawdown"),          higher_good=False)
    row("Daily P&L volatility", m.get("daily_pnl_volatility"), higher_good=False)
    row("Sharpe-like ratio",  m.get("sharpe_like"))

    print("\n  TREND")
    row("7-day rolling avg P&L",    m.get("rolling_7d_pnl"))
    row("Month-over-month change",  m.get("mom_change"))
    print("=" * 60 + "\n")


def calc_drawdown(series):
    peak = series.cummax()
    return (series / peak - 1) * 100

# ---------------------------------------------------------------------------
# 4. HTML dashboard generation
# ---------------------------------------------------------------------------

def generate_dashboard(df: pd.DataFrame, m: dict, output_path: str = "investment_dashboard.html"):
    df_c = df.dropna(subset=["overallPNL"]).copy()
    
    # ------------------------------------------------------------------
    # Equity curves (Growth of ₹100)
    # ------------------------------------------------------------------
    
    df_c["strategy_equity"] = (
        100 * (1 + df_c["overallPNL"] / 100).cumprod()
    )
    
    benchmarks = {
        "Nifty": "niftyReturns",
        "Sensex": "sensexReturns",
        "Bank Nifty": "niftyBankReturns",
        "Midcap": "niftyMidCapReturns",
        "Smallcap": "niftySmallCapReturns"
    }
    
    for name, col in benchmarks.items():
        if col in df_c.columns:
            df_c[f"{name}_equity"] = (
                100 * (1 + df_c[col].fillna(0) / 100).cumprod()
            )

    # ==========================================
    # ADD EVERYTHING BELOW HERE
    # ==========================================

    benchmark_summary = []

    strategy_final = df_c["strategy_equity"].iloc[-1]

    for name in benchmarks:

        benchmark_final = df_c[f"{name}_equity"].iloc[-1]

        benchmark_summary.append({
            "Benchmark": name,
            "Return %": round(
                (benchmark_final / 100 - 1) * 100,
                2
            ),
            "Alpha %": round(
                (strategy_final / benchmark_final - 1) * 100,
                2
            ),
            "Max DD %": round(
                calc_drawdown(
                    df_c[f"{name}_equity"]
                ).min(),
                2
            )
        })

    strategy_return = round(
        (strategy_final / 100 - 1) * 100,
        2
    )

    strategy_dd = round(
        calc_drawdown(
            df_c["strategy_equity"]
        ).min(),
        2
    )

    strategy_equity = (
        df_c["strategy_equity"]
        .round(2)
        .tolist()
    )

    nifty_equity = (
        df_c["Nifty_equity"]
        .round(2)
        .tolist()
    )

    sensex_equity = (
        df_c["Sensex_equity"]
        .round(2)
        .tolist()
    )

    bank_equity = (
        df_c["Bank Nifty_equity"]
        .round(2)
        .tolist()
    )

    midcap_equity = (
        df_c["Midcap_equity"]
        .round(2)
        .tolist()
    )

    smallcap_equity = (
        df_c["Smallcap_equity"]
        .round(2)
        .tolist()
    )

    strategy_dd_curve = (
        calc_drawdown(
            df_c["strategy_equity"]
        )
        .round(2)
        .tolist()
    )

    nifty_dd_curve = (
        calc_drawdown(
            df_c["Nifty_equity"]
        )
        .round(2)
        .tolist()
    )

    midcap_dd_curve = (
        calc_drawdown(
            df_c["Midcap_equity"]
        )
        .round(2)
        .tolist()
    )

    smallcap_dd_curve = (
        calc_drawdown(
            df_c["Smallcap_equity"]
        )
        .round(2)
        .tolist()
    )

    benchmark_rows = ""

    for row in benchmark_summary:

        alpha_class = (
            "good"
            if row["Alpha %"] > 0
            else "bad"
        )

        benchmark_rows += f"""
        <tr>
            <td>{row['Benchmark']}</td>
            <td>{row['Return %']}%</td>
            <td class="{alpha_class}">
                {row['Alpha %']}%
            </td>
            <td>{row['Max DD %']}%</td>
        </tr>
        """

    
    df_c["cumulative_pnl"] = (
        100 * (1 + df_c["overallPNL"] / 100).cumprod()
    )
    
    dates = [d.strftime("%Y-%m-%d") for d in df_c["date"]]
    
    cum_pnl = (
        df_c["cumulative_pnl"]
        .round(2)
        .tolist()
    )
    
    daily_pnl = (
        df_c["overallPNL"]
        .round(6)
        .tolist()
    )

    monthly_labels = list(m["monthly_pnl"].keys())
    monthly_vals   = list(m["monthly_pnl"].values())

    bm_names   = ["Nifty", "Sensex", "Nifty Bank", "Nifty Midcap", "Nifty Smallcap"]
    bm_keys    = ["nifty", "sensex", "nifty_bank", "nifty_midcap", "nifty_smallcap"]
    alpha_vals = [round(m["alpha"].get(k, 0), 6) for k in bm_keys]

    def fmt(val, pct=False, decimals=4):
        if val is None:
            return "N/A"
        if pct:
            return f"{val * 100:.2f}%"
        return f"{val:+.{decimals}f}" if val != 0 else "0.0000"

    def color(val, higher_good=True):
        if val is None:
            return "neutral"
        return "good" if (val > 0) == higher_good else "bad"

    alpha_rows = ""
    for name, val in zip(bm_names, alpha_vals):
        if not m["alpha"]:
            break
        badge_cls = "badge-good" if val >= 0 else "badge-bad"
        label = "Outperforming" if val >= 0 else "Underperforming"
        val_cls = "good" if val >= 0 else "bad"
        alpha_rows += (
            f"<tr><td>{name}</td>"
            f"<td class='{val_cls}'>{val:+.4f}</td>"
            f"<td><span class='badge {badge_cls}'>{label}</span></td></tr>\n"
        )
    if not alpha_rows:
        alpha_rows = "<tr><td colspan='3' style='color:#888780;font-size:13px'>Benchmark data not yet in dataset — fill index return columns to see alpha.</td></tr>"

    generated = datetime.now().strftime("%B %d, %Y at %H:%M")
    n_days = len(df_c)

    print(len(strategy_equity))
    print(strategy_equity[:5])
    
    print(len(nifty_equity))
    print(nifty_equity[:5])
    
    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Investment dashboard</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;background:#f5f5f3;color:#1a1a18;padding:2rem;}}
h1{{font-size:22px;font-weight:500;margin-bottom:.25rem}}
.filter-bar{{display:flex;gap:8px;margin-bottom:1rem;flex-wrap:wrap}}
.filter-btn{{font-size:12px;padding:5px 14px;border:.5px solid #e0ded9;border-radius:20px;background:#fff;cursor:pointer;color:#636360}}
.filter-btn:hover{{background:#f5f5f3}}
.filter-btn.active{{background:#185FA5;color:#fff;border-color:#185FA5}}




.sub{{font-size:14px;color:#6b6b66;margin-bottom:2rem}}
.sec{{font-size:11px;font-weight:500;text-transform:uppercase;letter-spacing:.05em;color:#888780;margin:1.75rem 0 .75rem}}
.kpi-grid{{display:grid;grid-template-columns:repeat(auto-fit,minmax(160px,1fr));gap:10px}}
.kpi{{background:#fff;border:.5px solid #e0ded9;border-radius:10px;padding:1rem}}
.kpi-label{{font-size:12px;color:#888780;margin-bottom:4px}}
.kpi-value{{font-size:22px;font-weight:500}}
.kpi-sub{{font-size:12px;color:#888780;margin-top:2px}}
.good{{color:#3B6D11}}.bad{{color:#A32D2D}}.neutral{{color:#636360}}
.chart-grid{{display:grid;grid-template-columns:repeat(auto-fit,minmax(340px,1fr));gap:12px;margin-top:.75rem}}
.chart-card{{background:#fff;border:.5px solid #e0ded9;border-radius:10px;padding:1.25rem}}
.chart-title{{font-size:14px;font-weight:500;margin-bottom:1rem}}
table{{width:100%;border-collapse:collapse;font-size:13px}}
th{{text-align:left;font-weight:500;font-size:12px;color:#888780;padding:6px 0;border-bottom:.5px solid #e0ded9}}
td{{padding:8px 0;border-bottom:.5px solid #f0ede8}}
.badge{{font-size:11px;padding:2px 8px;border-radius:4px;display:inline-block}}
.badge-good{{background:#EAF3DE;color:#3B6D11}}
.badge-bad{{background:#FCEBEB;color:#A32D2D}}
footer{{font-size:12px;color:#888780;margin-top:2rem;padding-top:1rem;border-top:.5px solid #e0ded9}}
</style>
</head>
<body>

<h1>Portfolio performance dashboard</h1>
<p class="sub">Generated {generated} &nbsp;·&nbsp; {n_days} trading sessions</p>

<p class="sec">Return performance</p>
<div class="kpi-grid">
  <div class="kpi">
    <div class="kpi-label">Total Returns</div>
    <div class="kpi-value {color(m.get('total_pnl'))}">{fmt(m.get('total_pnl'))}</div>
    <div class="kpi-sub">Total across all days</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">Avg daily P&amp;L</div>
    <div class="kpi-value {color(m.get('avg_daily_pnl'))}">{fmt(m.get('avg_daily_pnl'))}</div>
    <div class="kpi-sub">Mean per session</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">Win rate</div>
    <div class="kpi-value {color(m.get('win_rate'), higher_good=True)}">{fmt(m.get('win_rate'), pct=True)}</div>
    <div class="kpi-sub">% days with positive P&amp;L</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">Return on holding fund</div>
    <div class="kpi-value {color(m.get('avg_return_on_holding'))}">{fmt(m.get('avg_return_on_holding'))}</div>
    <div class="kpi-sub">Avg % on deployed capital</div>
  </div>
</div>

<p class="sec">Risk metrics</p>
<div class="kpi-grid">
  <div class="kpi">
    <div class="kpi-label">Max drawdown</div>
    <div class="kpi-value bad">{fmt(m.get('max_drawdown'))}</div>
    <div class="kpi-sub">Worst cumulative drop from peak</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">Daily P&amp;L volatility</div>
    <div class="kpi-value neutral">{fmt(m.get('daily_pnl_volatility'))}</div>
    <div class="kpi-sub">Std dev of day-over-day change</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">Sharpe-like ratio</div>
    <div class="kpi-value {color(m.get('sharpe_like'))}">{fmt(m.get('sharpe_like'))}</div>
    <div class="kpi-sub">Avg return ÷ volatility</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">Capital utilization</div>
    <div class="kpi-value neutral">{fmt(m.get('avg_utilization'))}</div>
    <div class="kpi-sub">Avg non-idle capital</div>
  </div>
</div>

<div class="filter-bar" id="filterBar">
  <button class="filter-btn" data-months="1">1M</button>
  <button class="filter-btn" data-months="3">3M</button>
  <button class="filter-btn" data-months="6">6M</button>
  <button class="filter-btn" data-months="ytd">YTD</button>
  <button class="filter-btn" data-months="12">1Y</button>
  <button class="filter-btn active" data-months="all">All</button>
</div>


<p class="sec">Charts</p>
<div class="chart-grid">

  <div class="chart-card" style="grid-column:1/-1">
    <div class="chart-title">Total Returns over time</div>
    <div style="position:relative;height:220px">
      <canvas id="cChart" role="img" aria-label="Line chart of Total Returns over time">Total portfolio P&L trend.</canvas>
    </div>
  </div>

  <div class="chart-card">
    <div class="chart-title">Daily P&amp;L (green = gain, red = loss)</div>
    <div style="position:relative;height:200px">
      <canvas id="dChart" role="img" aria-label="Bar chart of daily P&L">Daily P&L bar chart, positive and negative values.</canvas>
    </div>
  </div>

  <div class="chart-card">
    <div class="chart-title">Monthly P&amp;L</div>
    <div style="position:relative;height:200px">
      <canvas id="mChart" role="img" aria-label="Bar chart of monthly P&L aggregates">Monthly P&L aggregated by calendar month.</canvas>
    </div>
  </div>

  <div class="chart-card" style="grid-column:1/-1">
    <div class="chart-title">Alpha vs benchmark indices</div>
    <table>
      <thead><tr><th>Index</th><th>Your alpha</th><th>Status</th></tr></thead>
      <tbody>{alpha_rows}</tbody>
    </table>
    <p style="font-size:12px;color:#888780;margin-top:.75rem">Alpha = compounded overallPNL return minus compounded benchmark return.Positive means the strategy outperformed the benchmark.</p>
  </div>

</div>

<p class="sec">Benchmark Comparison</p>

<div class="chart-grid">

    <div class="chart-card" style="grid-column:1/-1">

        <div class="chart-title">
            Growth of ₹100
        </div>

        <div style="position:relative;height:350px">
            <canvas id="benchmarkChart"></canvas>
        </div>

    </div>

    <div class="chart-card" style="grid-column:1/-1">

        <div class="chart-title">
            Drawdown Comparison
        </div>

        <div style="position:relative;height:350px">
            <canvas id="drawdownChart"></canvas>
        </div>

    </div>

    <div class="chart-card" style="grid-column:1/-1">

        <div class="chart-title">
            Benchmark Scorecard
        </div>

        <table>

            <thead>
                <tr>
                    <th>Benchmark</th>
                    <th>Total Return</th>
                    <th>Alpha vs Strategy</th>
                    <th>Max Drawdown</th>
                </tr>
            </thead>

            <tbody>

                <tr>
                    <td><b>Strategy</b></td>
                    <td><b>{strategy_return}%</b></td>
                    <td>-</td>
                    <td><b>{strategy_dd}%</b></td>
                </tr>

                {benchmark_rows}

            </tbody>

        </table>

    </div>

</div>

<footer>
  All return and benchmark metrics are calculated from overallPNL and benchmark return columns. Capital utilization metrics use fund deployment fields.
  N/A = insufficient data in that column. &nbsp;·&nbsp; Script: investment_metrics.py
</footer>

<script>
const allDates          = {json.dumps(dates)};
const allCumPNL         = {json.dumps(cum_pnl)};
const allDaily          = {json.dumps(daily_pnl)};
const allMLabels        = {json.dumps(monthly_labels)};
const allMVals          = {json.dumps(monthly_vals)};
const allStrategyEquity = {json.dumps(strategy_equity)};
const allNiftyEquity    = {json.dumps(nifty_equity)};
const allSensexEquity   = {json.dumps(sensex_equity)};
const allBankEquity     = {json.dumps(bank_equity)};
const allMidcapEquity   = {json.dumps(midcap_equity)};
const allSmallcapEquity = {json.dumps(smallcap_equity)};
const allStrategyDD     = {json.dumps(strategy_dd_curve)};
const allNiftyDD        = {json.dumps(nifty_dd_curve)};
const allMidcapDD       = {json.dumps(midcap_dd_curve)};
const allSmallcapDD     = {json.dumps(smallcap_dd_curve)};

const TICKS = {{ maxTicksLimit:8, maxRotation:30 }};
const GRID  = {{ color:'rgba(0,0,0,.05)' }};

/* ── helpers ─────────────────────────────────────────────────────── */
function reindex(arr) {{
  if (!arr || !arr.length || arr[0] === 0) return arr;
  const base = arr[0];
  return arr.map(v => parseFloat((v / base * 100).toFixed(2)));
}}

function getCutoff(months) {{
  const last = new Date(allDates[allDates.length - 1]);
  if (months === 'all') return null;
  if (months === 'ytd') return new Date(last.getFullYear(), 0, 1);
  const d = new Date(last);
  d.setMonth(d.getMonth() - parseInt(months, 10));
  return d;
}}

function sliceFrom(cutoff) {{
  if (!cutoff) return 0;
  const idx = allDates.findIndex(d => new Date(d) >= cutoff);
  return idx === -1 ? 0 : idx;
}}

function filterMonthly(cutoff) {{
  if (!cutoff) return {{ labels: allMLabels, vals: allMVals }};
  const cutStr = cutoff.getFullYear() + '-' +
                 String(cutoff.getMonth() + 1).padStart(2, '0');
  const idx = allMLabels.findIndex(l => l >= cutStr);
  if (idx === -1) return {{ labels: [], vals: [] }};
  return {{ labels: allMLabels.slice(idx), vals: allMVals.slice(idx) }};
}}

function ddFromEquity(equity) {{
  let peak = -Infinity;
  return equity.map(v => {{
    if (v > peak) peak = v;
    return parseFloat(((v / peak - 1) * 100).toFixed(2));
  }});
}}

/* ── chart instances ─────────────────────────────────────────────── */
const cChart = new Chart(document.getElementById('cChart'), {{
  type: 'line',
  data: {{ labels: [], datasets: [{{
    label: 'Total returns', borderColor: '#185FA5',
    backgroundColor: 'rgba(24,95,165,.07)',
    fill: true, borderWidth: 2, tension: .3, pointRadius: 0
  }}] }},
  options: {{
    responsive: true, maintainAspectRatio: false,
    plugins: {{ legend: {{ display: false }} }},
    scales: {{ x: {{ ticks: TICKS, grid: {{ display: false }} }}, y: {{ grid: GRID }} }}
  }}
}});

const dChart = new Chart(document.getElementById('dChart'), {{
  type: 'bar',
  data: {{ labels: [], datasets: [{{ label: 'Daily P&L', borderRadius: 2 }}] }},
  options: {{
    responsive: true, maintainAspectRatio: false,
    plugins: {{ legend: {{ display: false }} }},
    scales: {{ x: {{ ticks: TICKS, grid: {{ display: false }} }}, y: {{ grid: GRID }} }}
  }}
}});

const mChart = new Chart(document.getElementById('mChart'), {{
  type: 'bar',
  data: {{ labels: [], datasets: [{{ label: 'Monthly P&L', borderRadius: 3 }}] }},
  options: {{
    responsive: true, maintainAspectRatio: false,
    plugins: {{ legend: {{ display: false }} }},
    scales: {{
      x: {{ ticks: {{ autoSkip: false, maxRotation: 30 }}, grid: {{ display: false }} }},
      y: {{ grid: GRID }}
    }}
  }}
}});

const benchmarkChart = new Chart(document.getElementById('benchmarkChart'), {{
  type: 'line',
  data: {{ labels: [], datasets: [
    {{ label: 'Strategy',   borderWidth: 3, pointRadius: 0 }},
    {{ label: 'Nifty',      pointRadius: 0 }},
    {{ label: 'Sensex',     pointRadius: 0 }},
    {{ label: 'Bank Nifty', pointRadius: 0 }},
    {{ label: 'Midcap',     pointRadius: 0 }},
    {{ label: 'Smallcap',   pointRadius: 0 }}
  ] }},
  options: {{ responsive: true, maintainAspectRatio: false }}
}});

const drawdownChart = new Chart(document.getElementById('drawdownChart'), {{
  type: 'line',
  data: {{ labels: [], datasets: [
    {{ label: 'Strategy', borderWidth: 3, pointRadius: 0 }},
    {{ label: 'Nifty',    pointRadius: 0 }},
    {{ label: 'Midcap',   pointRadius: 0 }},
    {{ label: 'Smallcap', pointRadius: 0 }}
  ] }},
  options: {{ responsive: true, maintainAspectRatio: false }}
}});

/* ── main filter function ────────────────────────────────────────── */
function applyFilter(months) {{
  const cutoff = getCutoff(months);
  const i = sliceFrom(cutoff);

  /* daily slices */
  const dates  = allDates.slice(i);
  const cumPNL = allCumPNL.slice(i);
  const daily  = allDaily.slice(i);
  const {{ labels: mL, vals: mV }} = filterMonthly(cutoff);

  /* cumulative returns — re-index to 100 */
  cChart.data.labels = dates;
  cChart.data.datasets[0].data = reindex(cumPNL);
  cChart.update('none');

  /* daily bars */
  dChart.data.labels = dates;
  dChart.data.datasets[0].data = daily;
  dChart.data.datasets[0].backgroundColor =
    daily.map(v => v >= 0 ? 'rgba(59,109,17,.75)' : 'rgba(163,45,45,.75)');
  dChart.update('none');

  /* monthly bars */
  mChart.data.labels = mL;
  mChart.data.datasets[0].data = mV;
  mChart.data.datasets[0].backgroundColor =
    mV.map(v => v >= 0 ? 'rgba(59,109,17,.75)' : 'rgba(163,45,45,.75)');
  mChart.update('none');

  /* equity curves — all re-indexed to 100 at period start */
  const equitySeries = [
    allStrategyEquity, allNiftyEquity, allSensexEquity,
    allBankEquity, allMidcapEquity, allSmallcapEquity
  ];
  benchmarkChart.data.labels = dates;
  equitySeries.forEach((s, idx) => {{
    benchmarkChart.data.datasets[idx].data = reindex(s.slice(i));
  }});
  benchmarkChart.update('none');

  /* drawdown — recomputed fresh from re-indexed equity */
  const ddSeries = [
    allStrategyEquity, allNiftyEquity, allMidcapEquity, allSmallcapEquity
  ];
  drawdownChart.data.labels = dates;
  ddSeries.forEach((s, idx) => {{
    drawdownChart.data.datasets[idx].data = ddFromEquity(reindex(s.slice(i)));
  }});
  drawdownChart.update('none');
}}

/* ── button wiring ───────────────────────────────────────────────── */
document.getElementById('filterBar').addEventListener('click', function(e) {{
  const btn = e.target.closest('.filter-btn');
  if (!btn) return;
  document.querySelectorAll('.filter-btn').forEach(b => b.classList.remove('active'));
  btn.classList.add('active');
  applyFilter(btn.dataset.months);
}});

applyFilter('all');
</script>
</body>
</html>"""

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"Dashboard saved → {output_path}")

    import plotly.graph_objects as go
    fig = go.Figure()
    
    fig.add_trace(
        go.Scatter(
            x=df_c["date"],
            y=df_c["strategy_equity"],
            name="Strategy"
        )
    )
    
    fig.add_trace(
        go.Scatter(
            x=df_c["date"],
            y=df_c["Nifty_equity"],
            name="Nifty"
        )
    )
    
    fig.add_trace(
        go.Scatter(
            x=df_c["date"],
            y=df_c["Midcap_equity"],
            name="Midcap"
        )
    )
    
    fig.show()
    
    return output_path


# ---------------------------------------------------------------------------
# 5. Entry point
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    filepath = '/kaggle/working/daysPNL_dfUpdated_fixed.csv'
    output_path = "investment_dashboard.html"

    print(f"Loading: {filepath}")
    df = load_data(filepath)
    print(f"  {len(df)} rows  |  {df['date'].min().date()} → {df['date'].max().date()}")

    metrics = calculate_metrics(df)
    print_summary(metrics)
    generate_dashboard(df, metrics, output_path)

Loading: /kaggle/working/daysPNL_dfUpdated_fixed.csv
  544 rows  |  2024-09-04 → 2026-08-07

  PORTFOLIO METRICS SUMMARY

  RETURN PERFORMANCE
  Compounded Total Return (%)                   +23.3288  ✓
  Avg daily P&L                                  +0.0388  ✓
  Return on holding fund (avg)                   +0.0399  ✓
  Churn efficiency ratio                        +20.0528  ✓

  BENCHMARK ALPHA
  Alpha vs Nifty                                +22.3100  ✓
  Alpha vs Sensex                               +25.1700  ✓
  Alpha vs Nifty Bank                           +10.1500  ✓
  Alpha vs Nifty Midcap                         +12.7100  ✓
  Alpha vs Nifty Smallcap                       +20.2600  ✓

  CAPITAL EFFICIENCY
  Avg capital utilization                         65.09%  ✓
  Avg idle fund %                                 34.91%  ✗
  Return per ₹ deployed                          +0.0000  ✓

  RISK METRICS
  Win rate                                        41.73%  ✓
  Max drawdown      

In [53]:
df.tail()

,date,overallPNL,PNLonFundsChurned,totalAccountValue,totalIdleFund,percIdleFund,percPNL_onHoldingFund,Parag_Parikh_Flexi_Cap_Fund,QUANTMUTUALFUND_MF,SBIMutualFund_MF,...,MOTILALOSWAL_MF,BANDHANMUTUALFUND_MF,EDELWEISSMUTUALFUND_MF,HDFCMutualFund_MF,INVESCOMUTUALFUND_MidCap_MF,niftyReturns,sensexReturns,niftyBankReturns,niftyMidCapReturns,niftySmallCapReturns
539,2026-07-31 15:29:19.417175,0.000,0.000,586204.179988,214559.70,36.602,0.000,0.777293,0.554314,0.789197,...,0.389464,0.317996,0.410912,0.317509,0.124886,0.273264,0.213646,0.205346,0.465455,0.416349
540,2026-08-03 15:29:30.774270,0.224,1.400,586149.579988,252214.52,43.029,0.394,1.687838,1.321968,0.801873,...,1.567270,1.310935,1.129662,1.306705,2.041410,1.602306,0.697090,1.716760,1.212268,1.376961
541,2026-08-05 15:35:18.989766,0.042,0.853,638469.989988,361066.40,56.552,0.097,-0.561926,0.412880,0.525313,...,0.330101,0.615635,0.195210,0.286437,0.053338,0.039610,0.193870,-0.288824,0.238567,0.701593
542,2026-08-06 15:35:21.193872,0.000,0.000,638763.189988,360994.20,56.515,0.000,-0.565101,-0.313158,0.241545,...,0.268376,0.212747,-0.238724,-0.035596,-0.213237,0.046092,0.475637,0.560617,-0.396950,0.200306
543,2026-08-07 15:35:23.286474,0.000,0.000,638763.189988,320770.30,50.217,0.000,0.020150,0.605441,-0.105972,...,0.178551,0.007018,0.131998,0.349308,0.041095,-0.265262,-0.577027,-0.546297,0.224576,-0.011439


In [54]:
df["equity_curve"] = (
    100 * (1 + df["overallPNL"] / 100).cumprod()
)

In [55]:
df[["date", "overallPNL", "equity_curve"]]

,date,overallPNL,equity_curve
0,2024-09-04 13:51:19.000000,0.050,100.050000
1,2024-09-05 15:30:06.000000,0.004,100.054002
2,2024-09-06 15:30:07.000000,0.117,100.171065
3,2024-09-09 15:29:37.000000,0.063,100.234173
4,2024-09-10 15:29:58.000000,0.140,100.374501
...,...,...,...
539,2026-07-31 15:29:19.417175,0.000,123.001525
540,2026-08-03 15:29:30.774270,0.224,123.277049
541,2026-08-05 15:35:18.989766,0.042,123.328825
542,2026-08-06 15:35:21.193872,0.000,123.328825


In [56]:
df.head()

,date,overallPNL,PNLonFundsChurned,totalAccountValue,totalIdleFund,percIdleFund,percPNL_onHoldingFund,Parag_Parikh_Flexi_Cap_Fund,QUANTMUTUALFUND_MF,SBIMutualFund_MF,...,BANDHANMUTUALFUND_MF,EDELWEISSMUTUALFUND_MF,HDFCMutualFund_MF,INVESCOMUTUALFUND_MidCap_MF,niftyReturns,sensexReturns,niftyBankReturns,niftyMidCapReturns,niftySmallCapReturns,equity_curve
0,2024-09-04 13:51:19,0.050,NaN,NaN,NaN,NaN,NaN,-0.237925,-0.456558,-1.106998,...,0.223075,-0.008694,-0.121108,0.147726,-0.321007,-0.245653,-0.558822,-0.084194,0.308252,100.050000
1,2024-09-05 15:30:06,0.004,NaN,NaN,NaN,NaN,NaN,0.027840,-0.142595,-0.187424,...,0.667734,0.174761,0.766348,0.417091,-0.212709,-0.183941,0.141634,0.372454,0.953844,100.054002
2,2024-09-06 15:30:07,0.117,NaN,NaN,NaN,NaN,NaN,-1.267506,-1.394029,-2.310376,...,-1.024050,-1.294970,-0.870864,-0.795259,-1.165038,-1.237489,-1.741105,-1.370075,-1.013768,100.171065
3,2024-09-09 15:29:37,0.063,NaN,NaN,NaN,NaN,NaN,-0.100776,-0.148791,-0.808089,...,-0.591786,-0.362284,-0.478841,-0.224662,0.339005,0.462665,1.069560,-0.250971,-0.829972,100.234173
4,2024-09-10 15:29:58,0.140,NaN,NaN,NaN,NaN,NaN,0.701676,0.910216,0.200513,...,1.084171,0.720142,1.183577,1.115603,0.419868,0.443541,0.302243,1.073705,1.250833,100.374501


# 27 June

In [58]:
"""
investment_metrics.py
---------------------
Calculate portfolio evaluation metrics and generate an HTML dashboard.
Usage:
    python investment_metrics.py portfolio_data.csv
    python investment_metrics.py portfolio_data.csv my_dashboard.html
Expected columns (tab- or comma-separated):
    date, overallPNL, PNLonFundsChurned, totalAccountValue,
    totalIdleFund, percIdleFund, percPNL_onHoldingFund,
    niftyReturns, sensexReturns, niftyBankReturns,
    niftyMidCapReturns, niftySmallCapReturns
Requires: pandas, numpy  (pip install pandas numpy)
"""
import sys
import json
import numpy as np
import pandas as pd
from datetime import datetime


# ---------------------------------------------------------------------------
# 1. Data loading
# ---------------------------------------------------------------------------

def load_data(filepath: str) -> pd.DataFrame:
    sep = "\t" if filepath.endswith(".tsv") else ","
    df = pd.read_csv(filepath, sep=sep)
    df = _drop_index_column(df)
    df = _parse_date_column(df)
    df = df.sort_values("date").reset_index(drop=True)
    return df


def _drop_index_column(df: pd.DataFrame) -> pd.DataFrame:
    """Remove a leading integer-index column that pandas or Excel sometimes writes."""
    first = str(df.columns[0]).strip()
    if first in ("", "Unnamed: 0") or first.isdigit():
        df = df.iloc[:, 1:]
    elif pd.api.types.is_integer_dtype(df.iloc[:, 0]):
        expected = pd.Series(range(len(df)))
        if (df.iloc[:, 0].reset_index(drop=True) == expected).all():
            df = df.iloc[:, 1:]
    return df.reset_index(drop=True)


def _parse_date_column(df):
    df["date"] = pd.to_datetime(df["date"], format="mixed", errors="coerce")
    n_bad = df["date"].isna().sum()
    if n_bad:
        print(f"Warning: {n_bad} date value(s) could not be parsed.")
    df = df.dropna(subset=["date"])
    return df


# ---------------------------------------------------------------------------
# 2. Metric calculations
# ---------------------------------------------------------------------------

def _safe_mean(series: pd.Series):
    s = series.dropna()
    return float(s.mean()) if len(s) else None


def _safe_last(series: pd.Series):
    s = series.dropna()
    return float(s.iloc[-1]) if len(s) else None


def calculate_metrics(df: pd.DataFrame) -> dict:
    m = {}
    pnl = df["overallPNL"].dropna()

    # --- Absolute return --------------------------------------------------
    m["total_pnl"]     = ((1 + pnl / 100).prod() - 1) * 100
    m["avg_daily_pnl"] = float(pnl.mean()) if len(pnl) else None
    m["latest_pnl"]    = float(pnl.iloc[-1]) if len(pnl) else None

    hold = df.get("percPNL_onHoldingFund", pd.Series(dtype=float))
    m["avg_return_on_holding"]    = _safe_mean(hold)
    m["latest_return_on_holding"] = _safe_last(hold)

    churned = df.get("PNLonFundsChurned", pd.Series(dtype=float))
    mask = churned.notna() & pnl.notna() & (pnl != 0)
    m["avg_churn_efficiency"] = float((churned[mask] / pnl[mask]).mean()) if mask.sum() else None

    # --- Alpha vs benchmarks ----------------------------------------------
    benchmarks = {
        "nifty":         "niftyReturns",
        "sensex":        "sensexReturns",
        "nifty_bank":    "niftyBankReturns",
        "nifty_midcap":  "niftyMidCapReturns",
        "nifty_smallcap":"niftySmallCapReturns",
    }
    alpha = {}
    strategy_return = ((1 + pnl / 100).prod() - 1) * 100
    for label, col in benchmarks.items():
        if col not in df.columns:
            continue
        bm = df[col].dropna()
        if len(bm) == 0:
            continue
        bm_return = ((1 + bm / 100).prod() - 1) * 100
        alpha[label] = round(strategy_return - bm_return, 2)

    m["strategy_return"] = round(strategy_return, 2)
    m["alpha"] = alpha

    # --- Mutual fund alpha ------------------------------------------------
    mutual_funds = {
        "Parag Parikh Flexi": "Parag_Parikh_Flexi_Cap_Fund",
        "Quant MF":           "QUANTMUTUALFUND_MF",
        "SBI MF":             "SBIMutualFund_MF",
        "Motilal Oswal MF":   "MOTILALOSWAL_MF",
        "Bandhan MF":         "BANDHANMUTUALFUND_MF",
        "Edelweiss MF":       "EDELWEISSMUTUALFUND_MF",
        "HDFC MF":            "HDFCMutualFund_MF",
        "Invesco MidCap MF":  "INVESCOMUTUALFUND_MidCap_MF",
    }
    mf_alpha = {}
    for label, col in mutual_funds.items():
        if col not in df.columns:
            continue
        bm = df[col].dropna()
        if len(bm) == 0:
            continue
        bm_return = ((1 + bm / 100).prod() - 1) * 100
        mf_alpha[label] = round(strategy_return - bm_return, 2)
    m["mf_alpha"] = mf_alpha

    # --- Capital efficiency -----------------------------------------------
    idle_pct = df.get("percIdleFund", pd.Series(dtype=float))
    m["avg_idle_pct"]    = _safe_mean(idle_pct)
    m["latest_idle_pct"] = _safe_last(idle_pct)
    m["avg_utilization"] = (100 - _safe_mean(idle_pct)) if _safe_mean(idle_pct) is not None else None

    idle_fund = df.get("totalIdleFund", pd.Series(dtype=float))
    m["latest_idle_fund"] = _safe_last(idle_fund)

    if {"totalAccountValue", "totalIdleFund", "overallPNL"}.issubset(df.columns):
        vm = df["totalAccountValue"].notna() & df["totalIdleFund"].notna() & pnl.notna()
        if vm.sum():
            deployed = (df.loc[vm, "totalAccountValue"] - df.loc[vm, "totalIdleFund"]).replace(0, np.nan)
            m["return_per_deployed"] = float((pnl[vm] / deployed).mean())
    else:
        m["return_per_deployed"] = None

    # --- Risk metrics -----------------------------------------------------
    if len(pnl) > 1:
        m["daily_pnl_volatility"] = float(pnl.std())

        equity      = 100 * (1 + pnl / 100).cumprod()
        rolling_max = equity.cummax()
        drawdown    = (equity / rolling_max - 1) * 100

        m["max_drawdown"] = float(drawdown.min())
        m["win_rate"]     = float((pnl > 0).mean())
        m["loss_rate"]    = float((pnl < 0).mean())   # ← new
        m["flat_rate"]    = float((pnl == 0).mean())  # ← new

        if m["daily_pnl_volatility"] and m["daily_pnl_volatility"] != 0:
            m["sharpe_like"] = float(m["avg_daily_pnl"] / m["daily_pnl_volatility"])
        else:
            m["sharpe_like"] = None
    else:
        m["daily_pnl_volatility"] = m["max_drawdown"] = None
        m["win_rate"] = m["loss_rate"] = m["flat_rate"] = m["sharpe_like"] = None

    # --- Trend ------------------------------------------------------------
    m["rolling_7d_pnl"] = float(pnl.rolling(7).mean().iloc[-1]) if len(pnl) >= 7 else None

    df2 = df.copy()
    df2["month"] = df2["date"].dt.to_period("M")
    monthly = (
        df2.groupby("month")["overallPNL"]
           .apply(lambda x: ((1 + x / 100).prod() - 1) * 100)
    )
    m["monthly_pnl"] = {str(k): round(v, 6) for k, v in monthly.items()}
    m["mom_change"]  = float(monthly.iloc[-1] - monthly.iloc[-2]) if len(monthly) >= 2 else None

    return m


# ---------------------------------------------------------------------------
# 3. Terminal summary
# ---------------------------------------------------------------------------

def print_summary(m: dict):
    def row(label, val, pct=False, already_pct=False, higher_good=True):
        if val is None:
            print(f"  {label:<40}  {'N/A':>12}")
            return
        if already_pct:
            fval = f"{val:.2f}%"
        elif pct:
            fval = f"{val * 100:.2f}%"
        else:
            fval = f"{val:+.4f}"
        mark = "✓" if (val > 0) == higher_good else "✗"
        print(f"  {label:<40}  {fval:>12}  {mark}")

    print("\n" + "=" * 60)
    print("  PORTFOLIO METRICS SUMMARY")
    print("=" * 60)

    print("\n  RETURN PERFORMANCE")
    row("Compounded total return (%)", m.get("total_pnl"))
    row("Avg daily P&L",               m.get("avg_daily_pnl"))
    row("Return on holding fund (avg)", m.get("avg_return_on_holding"))
    row("Churn efficiency ratio",       m.get("avg_churn_efficiency"))

    print("\n  BENCHMARK ALPHA")
    for k, v in (m.get("alpha") or {}).items():
        row(f"Alpha vs {k.replace('_', ' ').title()}", v)

    print("\n  CAPITAL EFFICIENCY")
    row("Avg capital utilization", m.get("avg_utilization"), already_pct=True)
    row("Avg idle fund %",         m.get("avg_idle_pct"),    already_pct=True, higher_good=False)
    row("Return per ₹ deployed",   m.get("return_per_deployed"))

    print("\n  RISK METRICS")
    row("Win rate",             m.get("win_rate"),              pct=True)
    row("Loss rate",            m.get("loss_rate"),             pct=True, higher_good=False)
    row("Flat rate",            m.get("flat_rate"),             pct=True, higher_good=False)
    row("Max drawdown",         m.get("max_drawdown"),          higher_good=False)
    row("Daily P&L volatility", m.get("daily_pnl_volatility"), higher_good=False)
    row("Sharpe-like ratio",    m.get("sharpe_like"))

    print("\n  TREND")
    row("7-day rolling avg P&L",   m.get("rolling_7d_pnl"))
    row("Month-over-month change", m.get("mom_change"))
    print("=" * 60 + "\n")


def calc_drawdown(series):
    peak = series.cummax()
    return (series / peak - 1) * 100


# ---------------------------------------------------------------------------
# 4. HTML dashboard generation
# ---------------------------------------------------------------------------

def generate_dashboard(df: pd.DataFrame, m: dict, output_path: str = "investment_dashboard.html"):
    df_c = df.dropna(subset=["overallPNL"]).copy()

    # Equity curves (Growth of ₹100)
    df_c["strategy_equity"] = 100 * (1 + df_c["overallPNL"] / 100).cumprod()

    benchmarks = {
        "Nifty":      "niftyReturns",
        "Sensex":     "sensexReturns",
        "Bank Nifty": "niftyBankReturns",
        "Midcap":     "niftyMidCapReturns",
        "Smallcap":   "niftySmallCapReturns",
    }
    for name, col in benchmarks.items():
        if col in df_c.columns:
            df_c[f"{name}_equity"] = 100 * (1 + df_c[col].fillna(0) / 100).cumprod()

    # Mutual fund equity curves
    mf_cols = {
        "Parag Parikh Flexi": "Parag_Parikh_Flexi_Cap_Fund",
        "Quant MF":           "QUANTMUTUALFUND_MF",
        "SBI MF":             "SBIMutualFund_MF",
        "Motilal Oswal MF":   "MOTILALOSWAL_MF",
        "Bandhan MF":         "BANDHANMUTUALFUND_MF",
        "Edelweiss MF":       "EDELWEISSMUTUALFUND_MF",
        "HDFC MF":            "HDFCMutualFund_MF",
        "Invesco MidCap MF":  "INVESCOMUTUALFUND_MidCap_MF",
        "ICICI MF" : "ICICIPrudentialMutualFund_MF",
        "Birla SunLife MF" : "BirlaSunLifeMutualFund_MF",
        "Invesco MF": "INVESCOMUTUALFUND_MF"
    }
    active_mfs = {name: col for name, col in mf_cols.items() if col in df_c.columns}
    for name, col in active_mfs.items():
        df_c[f"mf_{name}_equity"] = 100 * (1 + df_c[col].fillna(0) / 100).cumprod()

    # Benchmark scorecard
    strategy_final = df_c["strategy_equity"].iloc[-1]
    benchmark_summary = []
    for name in benchmarks:
        bm_final = df_c[f"{name}_equity"].iloc[-1]
        benchmark_summary.append({
            "Benchmark": name,
            "Return %":  round((bm_final / 100 - 1) * 100, 2),
            "Alpha %":   round((strategy_final / bm_final - 1) * 100, 2),
            "Max DD %":  round(calc_drawdown(df_c[f"{name}_equity"]).min(), 2),
        })

    strategy_return   = round((strategy_final / 100 - 1) * 100, 2)
    strategy_dd       = round(calc_drawdown(df_c["strategy_equity"]).min(), 2)

    strategy_equity   = df_c["strategy_equity"].round(2).tolist()
    nifty_equity      = df_c["Nifty_equity"].round(2).tolist()
    sensex_equity     = df_c["Sensex_equity"].round(2).tolist()
    bank_equity       = df_c["Bank Nifty_equity"].round(2).tolist()
    midcap_equity     = df_c["Midcap_equity"].round(2).tolist()
    smallcap_equity   = df_c["Smallcap_equity"].round(2).tolist()

    strategy_dd_curve = calc_drawdown(df_c["strategy_equity"]).round(2).tolist()
    nifty_dd_curve    = calc_drawdown(df_c["Nifty_equity"]).round(2).tolist()
    midcap_dd_curve   = calc_drawdown(df_c["Midcap_equity"]).round(2).tolist()
    smallcap_dd_curve = calc_drawdown(df_c["Smallcap_equity"]).round(2).tolist()

    benchmark_rows = ""
    for row in benchmark_summary:
        alpha_cls = "good" if row["Alpha %"] > 0 else "bad"
        benchmark_rows += (
            f"<tr><td>{row['Benchmark']}</td>"
            f"<td>{row['Return %']}%</td>"
            f"<td class='{alpha_cls}'>{row['Alpha %']}%</td>"
            f"<td>{row['Max DD %']}%</td></tr>\n"
        )

    df_c["cumulative_pnl"] = 100 * (1 + df_c["overallPNL"] / 100).cumprod()
    dates        = [d.strftime("%Y-%m-%d") for d in df_c["date"]]
    cum_pnl      = df_c["cumulative_pnl"].round(2).tolist()
    daily_pnl    = df_c["overallPNL"].round(6).tolist()

    monthly_labels = list(m["monthly_pnl"].keys())
    monthly_vals   = list(m["monthly_pnl"].values())

    idle_fund_vals = (df_c["totalIdleFund"].fillna(0).round(2).tolist()
                      if "totalIdleFund" in df_c.columns else [0] * len(df_c))
    idle_pct_vals  = (df_c["percIdleFund"].fillna(0).round(2).tolist()
                      if "percIdleFund"  in df_c.columns else [0] * len(df_c))

    # Mutual fund equity arrays + scorecard
    mf_names_list  = list(active_mfs.keys())
    mf_equities    = [df_c[f"mf_{n}_equity"].round(2).tolist() for n in mf_names_list]
    mf_dd_curves   = [calc_drawdown(df_c[f"mf_{n}_equity"]).round(2).tolist() for n in mf_names_list]

    mf_scorecard_rows = ""
    for name in mf_names_list:
        mf_final  = df_c[f"mf_{name}_equity"].iloc[-1]
        mf_ret    = round((mf_final / 100 - 1) * 100, 2)
        mf_alpha_v = round((strategy_final / mf_final - 1) * 100, 2)
        mf_dd_v   = round(calc_drawdown(df_c[f"mf_{name}_equity"]).min(), 2)
        alpha_cls = "good" if mf_alpha_v > 0 else "bad"
        mf_scorecard_rows += (
            f"<tr><td>{name}</td>"
            f"<td>{mf_ret}%</td>"
            f"<td class='{alpha_cls}'>{mf_alpha_v}%</td>"
            f"<td>{mf_dd_v}%</td></tr>\n"
        )

    bm_names = ["Nifty", "Sensex", "Nifty Bank", "Nifty Midcap", "Nifty Smallcap"]
    bm_keys  = ["nifty", "sensex", "nifty_bank", "nifty_midcap", "nifty_smallcap"]
    alpha_vals = [round(m["alpha"].get(k, 0), 6) for k in bm_keys]

    def fmt(val, pct=False, decimals=4):
        if val is None:
            return "N/A"
        if pct:
            return f"{val * 100:.2f}%"
        return f"{val:+.{decimals}f}" if val != 0 else "0.0000"

    def color(val, higher_good=True):
        if val is None:
            return "neutral"
        return "good" if (val > 0) == higher_good else "bad"

    alpha_rows = ""
    for name, val in zip(bm_names, alpha_vals):
        if not m["alpha"]:
            break
        badge_cls = "badge-good" if val >= 0 else "badge-bad"
        status    = "Outperforming" if val >= 0 else "Underperforming"
        val_cls   = "good" if val >= 0 else "bad"
        alpha_rows += (
            f"<tr><td>{name}</td>"
            f"<td class='{val_cls}'>{val:+.4f}</td>"
            f"<td><span class='badge {badge_cls}'>{status}</span></td></tr>\n"
        )
    if not alpha_rows:
        alpha_rows = "<tr><td colspan='3' style='color:#888780;font-size:13px'>Benchmark data not yet in dataset.</td></tr>"

    generated = datetime.now().strftime("%B %d, %Y at %H:%M")
    n_days    = len(df_c)

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Investment dashboard</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;background:#f5f5f3;color:#1a1a18;padding:2rem;}}
h1{{font-size:22px;font-weight:500;margin-bottom:.25rem}}
.filter-bar{{display:flex;gap:8px;margin-bottom:1rem;flex-wrap:wrap}}
.filter-btn{{font-size:12px;padding:5px 14px;border:.5px solid #e0ded9;border-radius:20px;background:#fff;cursor:pointer;color:#636360}}
.filter-btn:hover{{background:#f5f5f3}}
.filter-btn.active{{background:#185FA5;color:#fff;border-color:#185FA5}}
.sub{{font-size:14px;color:#6b6b66;margin-bottom:2rem}}
.sec{{font-size:11px;font-weight:500;text-transform:uppercase;letter-spacing:.05em;color:#888780;margin:1.75rem 0 .75rem}}
.kpi-grid{{display:grid;grid-template-columns:repeat(auto-fit,minmax(160px,1fr));gap:10px}}
.kpi{{background:#fff;border:.5px solid #e0ded9;border-radius:10px;padding:1rem}}
.kpi-label{{font-size:12px;color:#888780;margin-bottom:4px}}
.kpi-value{{font-size:22px;font-weight:500}}
.kpi-sub{{font-size:12px;color:#888780;margin-top:2px}}
.good{{color:#3B6D11}}.bad{{color:#A32D2D}}.neutral{{color:#636360}}
.chart-grid{{display:grid;grid-template-columns:repeat(auto-fit,minmax(340px,1fr));gap:12px;margin-top:.75rem}}
.chart-card{{background:#fff;border:.5px solid #e0ded9;border-radius:10px;padding:1.25rem}}
.chart-title{{font-size:14px;font-weight:500;margin-bottom:1rem}}
table{{width:100%;border-collapse:collapse;font-size:13px}}
th{{text-align:left;font-weight:500;font-size:12px;color:#888780;padding:6px 0;border-bottom:.5px solid #e0ded9}}
td{{padding:8px 0;border-bottom:.5px solid #f0ede8}}
.badge{{font-size:11px;padding:2px 8px;border-radius:4px;display:inline-block}}
.badge-good{{background:#EAF3DE;color:#3B6D11}}
.badge-bad{{background:#FCEBEB;color:#A32D2D}}
footer{{font-size:12px;color:#888780;margin-top:2rem;padding-top:1rem;border-top:.5px solid #e0ded9}}
</style>
</head>
<body>
<h1>Portfolio performance dashboard</h1>
<p class="sub">Generated {generated} &nbsp;·&nbsp; {n_days} trading sessions</p>

<p class="sec">Return performance</p>
<div class="kpi-grid">
  <div class="kpi">
    <div class="kpi-label">Total returns</div>
    <div class="kpi-value {color(m.get('total_pnl'))}">{fmt(m.get('total_pnl'))}</div>
    <div class="kpi-sub">Compounded across all days</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">Avg daily P&amp;L</div>
    <div class="kpi-value {color(m.get('avg_daily_pnl'))}">{fmt(m.get('avg_daily_pnl'))}</div>
    <div class="kpi-sub">Mean per session</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">Win rate</div>
    <div class="kpi-value {color(m.get('win_rate'), higher_good=True)}">{fmt(m.get('win_rate'), pct=True)}</div>
    <div class="kpi-sub">% days with positive P&amp;L</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">Return on holding fund</div>
    <div class="kpi-value {color(m.get('avg_return_on_holding'))}">{fmt(m.get('avg_return_on_holding'))}</div>
    <div class="kpi-sub">Avg % on deployed capital</div>
  </div>
</div>

<p class="sec">Risk metrics</p>
<div class="kpi-grid">
  <div class="kpi">
    <div class="kpi-label">Max drawdown</div>
    <div class="kpi-value bad">{fmt(m.get('max_drawdown'))}</div>
    <div class="kpi-sub">Worst cumulative drop from peak</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">Daily P&amp;L volatility</div>
    <div class="kpi-value neutral">{fmt(m.get('daily_pnl_volatility'))}</div>
    <div class="kpi-sub">Std dev of daily P&amp;L</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">Sharpe-like ratio</div>
    <div class="kpi-value {color(m.get('sharpe_like'))}">{fmt(m.get('sharpe_like'))}</div>
    <div class="kpi-sub">Avg return ÷ volatility</div>
  </div>
  <div class="kpi">
    <div class="kpi-label">Capital utilization</div>
    <div class="kpi-value neutral">{fmt(m.get('avg_utilization'))}</div>
    <div class="kpi-sub">Avg non-idle capital</div>
  </div>
</div>

<div class="filter-bar" id="filterBar">
  <button class="filter-btn" data-months="1">1M</button>
  <button class="filter-btn" data-months="3">3M</button>
  <button class="filter-btn" data-months="6">6M</button>
  <button class="filter-btn" data-months="ytd">YTD</button>
  <button class="filter-btn" data-months="12">1Y</button>
  <button class="filter-btn active" data-months="all">All</button>
</div>

<p class="sec">Charts</p>
<div class="chart-grid">
  <div class="chart-card" style="grid-column:1/-1">
    <div class="chart-title">Total returns over time</div>
    <div style="position:relative;height:220px">
      <canvas id="cChart" role="img" aria-label="Line chart of total returns over time">Total portfolio return trend.</canvas>
    </div>
  </div>

  <div class="chart-card">
    <div class="chart-title">Daily P&amp;L (green = gain, red = loss)</div>
    <div style="position:relative;height:200px">
      <canvas id="dChart" role="img" aria-label="Bar chart of daily P&L">Daily P&L bar chart.</canvas>
    </div>
  </div>

  <div class="chart-card">
    <div class="chart-title">Monthly P&amp;L</div>
    <div style="position:relative;height:200px">
      <canvas id="mChart" role="img" aria-label="Bar chart of monthly P&L">Monthly P&L aggregated by calendar month.</canvas>
    </div>
  </div>

  <div class="chart-card">
    <div class="chart-title">Win / loss / flat days</div>
    <div style="position:relative;height:180px">
      <canvas id="winPieChart" role="img" aria-label="Doughnut chart showing win loss and flat day distribution">Trading day outcome distribution.</canvas>
    </div>
    <div id="pieLegend" style="display:flex;justify-content:center;gap:16px;margin-top:14px;font-size:12px;color:#636360;flex-wrap:wrap"></div>
  </div>

  <div class="chart-card" style="grid-column:1/-1">
    <div class="chart-title">Idle fund — daily (₹ amount &amp; % of portfolio)</div>
    <div style="position:relative;height:220px">
      <canvas id="idleChart" role="img" aria-label="Bar chart of daily idle fund amount with percentage overlay">Daily idle fund levels.</canvas>
    </div>
  </div>

  <div class="chart-card" style="grid-column:1/-1">
    <div class="chart-title">Alpha vs benchmark indices</div>
    <table>
      <thead><tr><th>Index</th><th>Your alpha</th><th>Status</th></tr></thead>
      <tbody>{alpha_rows}</tbody>
    </table>
    <p style="font-size:12px;color:#888780;margin-top:.75rem">Alpha = compounded overallPNL return minus compounded benchmark return. Positive means the strategy outperformed the benchmark.</p>
  </div>
</div>

<p class="sec">Benchmark comparison</p>
<div class="chart-grid">
  <div class="chart-card" style="grid-column:1/-1">
    <div class="chart-title">Growth of ₹100</div>
    <div style="position:relative;height:350px">
      <canvas id="benchmarkChart" role="img" aria-label="Line chart showing growth of 100 rupees for strategy vs benchmarks">Equity curve comparison.</canvas>
    </div>
  </div>
  <div class="chart-card" style="grid-column:1/-1">
    <div class="chart-title">Drawdown comparison</div>
    <div style="position:relative;height:350px">
      <canvas id="drawdownChart" role="img" aria-label="Line chart showing drawdown comparison across strategy and benchmarks">Drawdown curves.</canvas>
    </div>
  </div>
  <div class="chart-card" style="grid-column:1/-1">
    <div class="chart-title">Benchmark scorecard</div>
    <table>
      <thead>
        <tr><th>Benchmark</th><th>Total return</th><th>Alpha vs strategy</th><th>Max drawdown</th></tr>
      </thead>
      <tbody>
        <tr>
          <td><b>Strategy</b></td>
          <td><b>{strategy_return}%</b></td>
          <td>—</td>
          <td><b>{strategy_dd}%</b></td>
        </tr>
        {benchmark_rows}
      </tbody>
    </table>
  </div>
</div>

<p class="sec">Mutual fund comparison</p>
<div class="chart-grid">
  <div class="chart-card" style="grid-column:1/-1">
    <div class="chart-title">Growth of ₹100 — mutual funds vs strategy</div>
    <div style="position:relative;height:350px">
      <canvas id="mfChart" role="img" aria-label="Line chart comparing mutual fund equity curves vs strategy">MF equity curve comparison.</canvas>
    </div>
  </div>
  <div class="chart-card" style="grid-column:1/-1">
    <div class="chart-title">Drawdown — mutual funds vs strategy</div>
    <div style="position:relative;height:280px">
      <canvas id="mfDDChart" role="img" aria-label="Line chart showing drawdown for mutual funds vs strategy">MF drawdown curves.</canvas>
    </div>
  </div>
  <div class="chart-card" style="grid-column:1/-1">
    <div class="chart-title">Mutual fund scorecard</div>
    <table>
      <thead>
        <tr><th>Fund</th><th>Total return</th><th>Alpha vs strategy</th><th>Max drawdown</th></tr>
      </thead>
      <tbody>
        <tr>
          <td><b>Strategy</b></td>
          <td><b>{strategy_return}%</b></td>
          <td>—</td>
          <td><b>{strategy_dd}%</b></td>
        </tr>
        {mf_scorecard_rows}
      </tbody>
    </table>
  </div>
</div>

<footer>
  All return and benchmark metrics are calculated using compounded returns from overallPNL and index return columns.
  Capital utilization uses fund deployment fields. N/A = insufficient data. &nbsp;·&nbsp; Script: investment_metrics.py
</footer>

<script>
const allDates          = {json.dumps(dates)};
const allCumPNL         = {json.dumps(cum_pnl)};
const allDaily          = {json.dumps(daily_pnl)};
const allMLabels        = {json.dumps(monthly_labels)};
const allMVals          = {json.dumps(monthly_vals)};
const allStrategyEquity = {json.dumps(strategy_equity)};
const allNiftyEquity    = {json.dumps(nifty_equity)};
const allSensexEquity   = {json.dumps(sensex_equity)};
const allBankEquity     = {json.dumps(bank_equity)};
const allMidcapEquity   = {json.dumps(midcap_equity)};
const allSmallcapEquity = {json.dumps(smallcap_equity)};
const allIdleFund       = {json.dumps(idle_fund_vals)};
const allIdlePct        = {json.dumps(idle_pct_vals)};
const allMFNames        = {json.dumps(mf_names_list)};
const allMFEquity       = {json.dumps(mf_equities)};
const allMFDD           = {json.dumps(mf_dd_curves)};
const allStrategyDD     = {json.dumps(strategy_dd_curve)};
const allNiftyDD        = {json.dumps(nifty_dd_curve)};
const allMidcapDD       = {json.dumps(midcap_dd_curve)};
const allSmallcapDD     = {json.dumps(smallcap_dd_curve)};

const TICKS = {{ maxTicksLimit:8, maxRotation:30 }};
const GRID  = {{ color:'rgba(0,0,0,.05)' }};

/* ── helpers ─────────────────────────────────────────────────────── */
function reindex(arr) {{
  if (!arr || !arr.length || arr[0] === 0) return arr;
  const base = arr[0];
  return arr.map(v => parseFloat((v / base * 100).toFixed(2)));
}}

function getCutoff(months) {{
  const last = new Date(allDates[allDates.length - 1]);
  if (months === 'all') return null;
  if (months === 'ytd') return new Date(last.getFullYear(), 0, 1);
  const d = new Date(last);
  d.setMonth(d.getMonth() - parseInt(months, 10));
  return d;
}}

function sliceFrom(cutoff) {{
  if (!cutoff) return 0;
  const idx = allDates.findIndex(d => new Date(d) >= cutoff);
  return idx === -1 ? 0 : idx;
}}

function filterMonthly(cutoff) {{
  if (!cutoff) return {{ labels: allMLabels, vals: allMVals }};
  const cutStr = cutoff.getFullYear() + '-' +
                 String(cutoff.getMonth() + 1).padStart(2, '0');
  const idx = allMLabels.findIndex(l => l >= cutStr);
  if (idx === -1) return {{ labels: [], vals: [] }};
  return {{ labels: allMLabels.slice(idx), vals: allMVals.slice(idx) }};
}}

function ddFromEquity(equity) {{
  let peak = -Infinity;
  return equity.map(v => {{
    if (v > peak) peak = v;
    return parseFloat(((v / peak - 1) * 100).toFixed(2));
  }});
}}

/* ── chart instances ─────────────────────────────────────────────── */
const cChart = new Chart(document.getElementById('cChart'), {{
  type: 'line',
  data: {{ labels: [], datasets: [{{
    label: 'Total returns', borderColor: '#185FA5',
    backgroundColor: 'rgba(24,95,165,.07)',
    fill: true, borderWidth: 2, tension: .3, pointRadius: 0
  }}] }},
  options: {{
    responsive: true, maintainAspectRatio: false,
    plugins: {{ legend: {{ display: false }} }},
    scales: {{ x: {{ ticks: TICKS, grid: {{ display: false }} }}, y: {{ grid: GRID }} }}
  }}
}});

const dChart = new Chart(document.getElementById('dChart'), {{
  type: 'bar',
  data: {{ labels: [], datasets: [{{ label: 'Daily P&L', borderRadius: 2 }}] }},
  options: {{
    responsive: true, maintainAspectRatio: false,
    plugins: {{ legend: {{ display: false }} }},
    scales: {{ x: {{ ticks: TICKS, grid: {{ display: false }} }}, y: {{ grid: GRID }} }}
  }}
}});

const mChart = new Chart(document.getElementById('mChart'), {{
  type: 'bar',
  data: {{ labels: [], datasets: [{{ label: 'Monthly P&L', borderRadius: 3 }}] }},
  options: {{
    responsive: true, maintainAspectRatio: false,
    plugins: {{ legend: {{ display: false }} }},
    scales: {{
      x: {{ ticks: {{ autoSkip: false, maxRotation: 30 }}, grid: {{ display: false }} }},
      y: {{ grid: GRID }}
    }}
  }}
}});

const winPieChart = new Chart(document.getElementById('winPieChart'), {{
  type: 'doughnut',
  data: {{
    labels: ['Win', 'Loss', 'Flat'],
    datasets: [{{
      data: [],
      backgroundColor: [
        'rgba(59,109,17,.85)',
        'rgba(163,45,45,.85)',
        'rgba(136,135,128,.75)'
      ],
      borderWidth: 0,
      hoverOffset: 6
    }}]
  }},
  options: {{
    responsive: true,
    maintainAspectRatio: false,
    cutout: '62%',
    plugins: {{ legend: {{ display: false }} }}
  }}
}});

const idleChart = new Chart(document.getElementById('idleChart'), {{
  type: 'bar',
  data: {{ labels: [], datasets: [
    {{
      label: 'Idle fund (₹)',
      borderRadius: 2,
      backgroundColor: 'rgba(255,165,0,.65)',
      yAxisID: 'yAmt'
    }},
    {{
      label: 'Idle %',
      type: 'line',
      borderColor: 'rgba(163,45,45,.85)',
      backgroundColor: 'transparent',
      borderWidth: 1.5,
      pointRadius: 0,
      tension: .3,
      yAxisID: 'yPct'
    }}
  ] }},
  options: {{
    responsive: true,
    maintainAspectRatio: false,
    plugins: {{ legend: {{ position: 'top', labels: {{ boxWidth: 10, font: {{ size: 11 }} }} }} }},
    scales: {{
      x: {{ ticks: TICKS, grid: {{ display: false }} }},
      yAmt: {{
        position: 'left',
        grid: GRID,
        title: {{ display: true, text: '₹ idle', font: {{ size: 11 }}, color: '#888780' }}
      }},
      yPct: {{
        position: 'right',
        grid: {{ display: false }},
        title: {{ display: true, text: '% idle', font: {{ size: 11 }}, color: '#888780' }},
        ticks: {{ callback: v => v + '%' }}
      }}
    }}
  }}
}});

/* MF palette — distinct colors for up to 8 funds */
const MF_COLORS = [
  '#E07B39','#9B59B6','#1ABC9C','#E74C3C',
  '#3498DB','#F39C12','#2ECC71','#95A5A6'
];

const mfDatasets = allMFNames.map((name, idx) => ({{
  label: name,
  borderColor: MF_COLORS[idx % MF_COLORS.length],
  backgroundColor: 'transparent',
  borderWidth: 1.5,
  pointRadius: 0,
  tension: .3,
  data: []
}}));

const mfChart = new Chart(document.getElementById('mfChart'), {{
  type: 'line',
  data: {{ labels: [], datasets: [
    {{ label: 'Strategy', borderColor: '#185FA5', borderWidth: 3, pointRadius: 0, tension: .3, data: [] }},
    ...mfDatasets
  ] }},
  options: {{ responsive: true, maintainAspectRatio: false,
    plugins: {{ legend: {{ position: 'top', labels: {{ boxWidth: 10, font: {{ size: 11 }} }} }} }}
  }}
}});

const mfDDDatasets = allMFNames.map((name, idx) => ({{
  label: name,
  borderColor: MF_COLORS[idx % MF_COLORS.length],
  backgroundColor: 'transparent',
  borderWidth: 1.5,
  pointRadius: 0,
  tension: .3,
  data: []
}}));

const mfDDChart = new Chart(document.getElementById('mfDDChart'), {{
  type: 'line',
  data: {{ labels: [], datasets: [
    {{ label: 'Strategy', borderColor: '#185FA5', borderWidth: 3, pointRadius: 0, tension: .3, data: [] }},
    ...mfDDDatasets
  ] }},
  options: {{ responsive: true, maintainAspectRatio: false,
    plugins: {{ legend: {{ position: 'top', labels: {{ boxWidth: 10, font: {{ size: 11 }} }} }} }}
  }}
}});

const benchmarkChart = new Chart(document.getElementById('benchmarkChart'), {{
  type: 'line',
  data: {{ labels: [], datasets: [
    {{ label: 'Strategy',   borderWidth: 3, pointRadius: 0 }},
    {{ label: 'Nifty',      pointRadius: 0 }},
    {{ label: 'Sensex',     pointRadius: 0 }},
    {{ label: 'Bank Nifty', pointRadius: 0 }},
    {{ label: 'Midcap',     pointRadius: 0 }},
    {{ label: 'Smallcap',   pointRadius: 0 }}
  ] }},
  options: {{ responsive: true, maintainAspectRatio: false }}
}});

const drawdownChart = new Chart(document.getElementById('drawdownChart'), {{
  type: 'line',
  data: {{ labels: [], datasets: [
    {{ label: 'Strategy', borderWidth: 3, pointRadius: 0 }},
    {{ label: 'Nifty',    pointRadius: 0 }},
    {{ label: 'Midcap',   pointRadius: 0 }},
    {{ label: 'Smallcap', pointRadius: 0 }}
  ] }},
  options: {{ responsive: true, maintainAspectRatio: false }}
}});

/* ── main filter function ────────────────────────────────────────── */
function applyFilter(months) {{
  const cutoff = getCutoff(months);
  const i = sliceFrom(cutoff);

  const dates  = allDates.slice(i);
  const cumPNL = allCumPNL.slice(i);
  const daily  = allDaily.slice(i);
  const {{ labels: mL, vals: mV }} = filterMonthly(cutoff);

  /* cumulative returns — re-index to 100 */
  cChart.data.labels = dates;
  cChart.data.datasets[0].data = reindex(cumPNL);
  cChart.update('none');

  /* daily bars */
  dChart.data.labels = dates;
  dChart.data.datasets[0].data = daily;
  dChart.data.datasets[0].backgroundColor =
    daily.map(v => v >= 0 ? 'rgba(59,109,17,.75)' : 'rgba(163,45,45,.75)');
  dChart.update('none');

  /* monthly bars */
  mChart.data.labels = mL;
  mChart.data.datasets[0].data = mV;
  mChart.data.datasets[0].backgroundColor =
    mV.map(v => v >= 0 ? 'rgba(59,109,17,.75)' : 'rgba(163,45,45,.75)');
  mChart.update('none');

  /* win / loss / flat pie — recomputed from filtered daily data */
  const winDays   = daily.filter(v => v >  0).length;
  const lossDays  = daily.filter(v => v <  0).length;
  const flatDays  = daily.filter(v => v === 0).length;
  const totalDays = daily.length || 1;

  winPieChart.data.datasets[0].data = [winDays, lossDays, flatDays];
  winPieChart.update('none');

  const pct = n => (n / totalDays * 100).toFixed(1) + '%';
  document.getElementById('pieLegend').innerHTML = [
    ['rgba(59,109,17,.85)',   'Win',  winDays,  pct(winDays)],
    ['rgba(163,45,45,.85)',   'Loss', lossDays, pct(lossDays)],
    ['rgba(136,135,128,.75)', 'Flat', flatDays, pct(flatDays)]
  ].map(([clr, lbl, cnt, p]) =>
    `<span style="display:flex;align-items:center;gap:5px">
       <span style="width:10px;height:10px;border-radius:2px;background:${{clr}}"></span>
       <span><strong style="color:#1a1a18">${{p}}</strong> ${{lbl}} (${{cnt}}d)</span>
     </span>`
  ).join('');

  /* idle fund bars + % line */
  idleChart.data.labels = dates;
  idleChart.data.datasets[0].data = allIdleFund.slice(i);
  idleChart.data.datasets[1].data = allIdlePct.slice(i);
  idleChart.update('none');

  /* mutual fund equity curves */
  mfChart.data.labels = dates;
  mfChart.data.datasets[0].data = reindex(allStrategyEquity.slice(i));
  allMFEquity.forEach((eq, idx) => {{
    mfChart.data.datasets[idx + 1].data = reindex(eq.slice(i));
  }});
  mfChart.update('none');

  /* mutual fund drawdown curves */
  mfDDChart.data.labels = dates;
  mfDDChart.data.datasets[0].data = ddFromEquity(reindex(allStrategyEquity.slice(i)));
  allMFEquity.forEach((eq, idx) => {{
    mfDDChart.data.datasets[idx + 1].data = ddFromEquity(reindex(eq.slice(i)));
  }});
  mfDDChart.update('none');

  /* equity curves — re-indexed to 100 at period start */
  const equitySeries = [
    allStrategyEquity, allNiftyEquity, allSensexEquity,
    allBankEquity, allMidcapEquity, allSmallcapEquity
  ];
  benchmarkChart.data.labels = dates;
  equitySeries.forEach((s, idx) => {{
    benchmarkChart.data.datasets[idx].data = reindex(s.slice(i));
  }});
  benchmarkChart.update('none');

  /* drawdown — recomputed fresh from re-indexed equity */
  const ddSeries = [
    allStrategyEquity, allNiftyEquity, allMidcapEquity, allSmallcapEquity
  ];
  drawdownChart.data.labels = dates;
  ddSeries.forEach((s, idx) => {{
    drawdownChart.data.datasets[idx].data = ddFromEquity(reindex(s.slice(i)));
  }});
  drawdownChart.update('none');
}}

/* ── button wiring ───────────────────────────────────────────────── */
document.getElementById('filterBar').addEventListener('click', function(e) {{
  const btn = e.target.closest('.filter-btn');
  if (!btn) return;
  document.querySelectorAll('.filter-btn').forEach(b => b.classList.remove('active'));
  btn.classList.add('active');
  applyFilter(btn.dataset.months);
}});

applyFilter('all');
</script>
</body>
</html>"""

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"Dashboard saved → {output_path}")

    import plotly.graph_objects as go
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df_c["date"], y=df_c["strategy_equity"], name="Strategy"))
    fig.add_trace(go.Scatter(x=df_c["date"], y=df_c["Nifty_equity"],    name="Nifty"))
    fig.add_trace(go.Scatter(x=df_c["date"], y=df_c["Midcap_equity"],   name="Midcap"))
    fig.show()

    return output_path


# ---------------------------------------------------------------------------
# 5. Entry point
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    filepath    = '/kaggle/working/daysPNL_dfUpdated_fixed.csv'
    output_path = "investment_dashboard.html"

    print(f"Loading: {filepath}")
    df = load_data(filepath)
    print(f"  {len(df)} rows  |  {df['date'].min().date()} → {df['date'].max().date()}")

    metrics = calculate_metrics(df)
    print_summary(metrics)
    generate_dashboard(df, metrics, output_path)     



Loading: /kaggle/working/daysPNL_dfUpdated_fixed.csv
  544 rows  |  2024-09-04 → 2026-08-07

  PORTFOLIO METRICS SUMMARY

  RETURN PERFORMANCE
  Compounded total return (%)                   +23.3288  ✓
  Avg daily P&L                                  +0.0388  ✓
  Return on holding fund (avg)                   +0.0399  ✓
  Churn efficiency ratio                        +20.0528  ✓

  BENCHMARK ALPHA
  Alpha vs Nifty                                +22.3100  ✓
  Alpha vs Sensex                               +25.1700  ✓
  Alpha vs Nifty Bank                           +10.1500  ✓
  Alpha vs Nifty Midcap                         +12.7100  ✓
  Alpha vs Nifty Smallcap                       +20.2600  ✓

  CAPITAL EFFICIENCY
  Avg capital utilization                         65.09%  ✓
  Avg idle fund %                                 34.91%  ✗
  Return per ₹ deployed                          +0.0000  ✓

  RISK METRICS
  Win rate                                        41.73%  ✓
  Loss rate         